# AI Company Health & Valuation Predictor

## Project overview

### Problem statement
Analyzing a company's financial health and estimating its market capitalization requires examining multiple financial indicators and market data.

### Proposed solution
A machine-learning system that builds a point-in-time dataset from company financial statements, predicts market capitalization, explains the factors behind its predictions, and produces a separate rule-based financial health assessment.

### Objectives
- Collect and validate financial statements and market data for multiple companies.
- Engineer core financial features (size, profitability, liquidity, leverage and revenue growth).
- Train and compare regression models using chronological validation.
- Implement Ridge regression optimization using gradient descent from scratch.
- Evaluate predictions and explain the learned weights.
- Generate a transparent company health score and export reusable model artifacts.

**Scope:** This streamlined version removes standalone calculus demonstrations and the extra calculus-derived financial features. Calculus remains where it is genuinely required: deriving and verifying the gradient, understanding the Hessian and convexity, and optimizing the valuation model with gradient descent.

> **Run order:** Run the notebook from top to bottom. Part 2 builds the multi-company dataset; Parts 3–5 train, evaluate and package the predictor; Part 6 is the gradient-descent optimization study. The first run needs internet access to Yahoo Finance and may take several minutes. Data downloads are cached in `data_cache/`.

---

# PART 2 — Building a Multi-Company, Leak-Free Dataset

Up to now the notebook used one company (Apple). A machine-learning model needs **many companies and many dates**. This part builds a table with one row per *company × fiscal year*.

## 2.1 What exactly are we predicting? (valuation target)

Three different "values" are often confused. They are **not interchangeable**:

| Concept | Definition | Observable? | Used here? |
|---|---|---|---|
| **Market capitalization** | share price × shares outstanding = what the stock market currently says the *equity* is worth | Yes, daily | **Target** |
| **Enterprise value (EV)** | market cap + total debt − cash = the market's value of the *whole operating business* (equity + debt holders). Simplified here: minority interest, preferred stock and leases are ignored | Yes (needs the balance sheet) | Computed and stored for comparison only |
| **Intrinsic value** | an analyst's estimate of what the business is *really* worth, e.g. from discounted future cash flows (DCF) | **No** — depends on forecasts and assumptions | Not modelled |

**Target = `log(market capitalization)` on the prediction date.**

* *Why market cap and not EV?* Debt and cash are also model inputs. Predicting EV would let the model "cheat" through the identity EV = market cap + debt − cash.
* *Why logarithm?* Market caps range from billions to trillions. On a log scale an error of 0.1 means roughly a 10% mistake for any company size, and linear models behave better.
* *What does the model really learn?* **"What market cap do companies with fundamentals like these typically have?"** — a statistical benchmark of market pricing, *not* the true or intrinsic value.

## 2.2 Preventing data leakage (the rules this notebook follows)

1. **Reporting lag.** A fiscal year ending on date *b* is assumed to become public **90 days later** (`REPORT_LAG_DAYS`). The prediction date is the first trading day on/after *b + 90 days*. yfinance does not provide filing dates, so 90 days is a deliberately conservative assumption.
2. **Features use only the past.** Every feature is computed from the fiscal years up to the one being scored. Derivatives use **backward** differences (this year vs. earlier years), never a "next year" value.
3. **No price-based features.** Prices appear only in the target.
4. **Chronological splits.** Training rows are earlier in time than validation rows, which are earlier than test rows. Preprocessing (imputation, outlier clipping, scaling) is fitted on training data only, inside a scikit-learn `Pipeline`.
5. **Automated test.** A cell below *changes the future statements* of some companies and verifies that earlier rows do not change.

## 2.3 Known data limitations (read before trusting any result)

* Yahoo Finance (via the unofficial `yfinance` package) typically provides only **about 4 annual statements per company**, so each company contributes only a few rows and the dataset is **small**.
* Yahoo shows the **latest restated** figures, not the numbers exactly as first published.
* The company list is today's large, successful companies → **survivorship bias**. Prices span roughly one market regime.
* Share counts and splits: Yahoo prices are adjusted for later stock splits, but reported share counts are not always. The code adjusts share counts for later splits and **checks the result against Yahoo's current market cap**; suspicious companies are excluded automatically.
* Only US-listed companies that report in USD are supported. Banks/insurers are left out because their statements do not have items such as gross profit or current assets.

The reusable code lives in a separate file so that a future API can import exactly the same logic. The next cell **writes that file** (`valuation_pipeline.py`). If you edit it later, restart the runtime before re-running.

In [ ]:
# Install the external data source library (run once if it is not already installed)
%pip -q install yfinance

import yfinance as yf
print("yfinance is ready.")

In [ ]:
%%writefile valuation_pipeline.py
"""
valuation_pipeline.py
=====================
Reusable data + feature code for the AI Company Health & Valuation Predictor.

This file is written by the notebook (cell "%%writefile valuation_pipeline.py") and is
imported both by the notebook and, later, by a Python API. It contains NO model training.

Design rules (see the notebook for the full explanation)
--------------------------------------------------------
* Every feature for a prediction date is computed from financial statements whose
  fiscal year ended at least REPORT_LAG_DAYS before that date (point-in-time).
* Prices are used ONLY to build the target (market capitalization), never as features.
"""

import pickle
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# --------------------------------------------------------------------------------------
# Configuration
# --------------------------------------------------------------------------------------
REPORT_LAG_DAYS = 90        # assume annual statements become public 90 days after year end
MAX_PRICE_GAP_DAYS = 10     # skip a row if no trading day exists within 10 days of the date
MIN_HISTORY = 2             # need the current and the previous fiscal year

BASE_NUMERIC = [
    "log_revenue", "log_total_assets",
    "gross_margin", "operating_margin", "net_margin", "fcf_margin", "roa",
    "current_ratio", "cash_to_assets", "debt_to_assets", "liabilities_to_assets",
    "revenue_growth",
]
CATEGORICAL = ["sector"]

# yfinance row names differ between companies, so each item has a list of candidates.
SOURCES = {
    "revenue":            ("income",   ["Total Revenue", "Operating Revenue"]),
    "gross_profit":       ("income",   ["Gross Profit"]),
    "operating_income":   ("income",   ["Operating Income", "Total Operating Income As Reported"]),
    "net_income":         ("income",   ["Net Income", "Net Income Common Stockholders"]),
    "interest_expense":   ("income",   ["Interest Expense", "Interest Expense Non Operating"]),
    "total_assets":       ("balance",  ["Total Assets"]),
    "total_liabilities":  ("balance",  ["Total Liabilities Net Minority Interest"]),
    "equity":             ("balance",  ["Stockholders Equity", "Common Stock Equity",
                                        "Total Equity Gross Minority Interest"]),
    "current_assets":     ("balance",  ["Current Assets"]),
    "current_liabilities": ("balance", ["Current Liabilities"]),
    "cash":               ("balance",  ["Cash Cash Equivalents And Short Term Investments",
                                        "Cash And Cash Equivalents"]),
    "total_debt":         ("balance",  ["Total Debt"]),
    "shares":             ("balance",  ["Ordinary Shares Number", "Share Issued"]),
    "operating_cash_flow": ("cashflow", ["Operating Cash Flow",
                                         "Cash Flow From Continuing Operating Activities"]),
    "free_cash_flow":     ("cashflow", ["Free Cash Flow"]),
    "capex":              ("cashflow", ["Capital Expenditure"]),
}


class DataUnavailableError(Exception):
    """Raised when usable data for a company cannot be obtained."""


# --------------------------------------------------------------------------------------
# 1. Downloading and caching raw data
# --------------------------------------------------------------------------------------
def _validate_raw(raw, ticker):
    if raw["prices"] is None or len(raw["prices"]) == 0 or "Close" not in raw["prices"]:
        raise DataUnavailableError(f"{ticker}: no price history returned")
    for key in ("income", "balance"):
        frame = raw[key]
        if frame is None or len(frame) == 0 or frame.dropna(how="all").shape[1] == 0:
            raise DataUnavailableError(f"{ticker}: empty {key} statement")


def fetch_raw(ticker, cache_dir="data_cache", refresh=False, retries=3, pause=1.0):
    """Download prices, statements, splits and basic info for one ticker (cached to disk)."""
    import yfinance as yf

    ticker = ticker.upper().strip()
    cache_path = Path(cache_dir) / f"{ticker}.pkl"
    if cache_path.exists() and not refresh:
        with open(cache_path, "rb") as handle:
            return pickle.load(handle)

    last_error = None
    for attempt in range(1, retries + 1):
        try:
            tk = yf.Ticker(ticker)
            prices = tk.history(period="max", auto_adjust=False)
            income, balance, cashflow = tk.income_stmt, tk.balance_sheet, tk.cashflow
            try:
                splits = tk.splits
            except Exception:
                splits = pd.Series(dtype=float)
            try:
                info = tk.info or {}
            except Exception:
                info = {}
            keep = ("sector", "industry", "currency", "financialCurrency",
                    "marketCap", "shortName", "longName", "quoteType")
            raw = {
                "ticker": ticker,
                "prices": prices,
                "income": income,
                "balance": balance,
                "cashflow": cashflow,
                "splits": splits,
                "info": {k: info.get(k) for k in keep},
                "downloaded_at": pd.Timestamp.now().isoformat(),
            }
            _validate_raw(raw, ticker)
            cache_path.parent.mkdir(parents=True, exist_ok=True)
            with open(cache_path, "wb") as handle:
                pickle.dump(raw, handle)
            return raw
        except Exception as exc:                       # network, rate limit, empty data ...
            last_error = exc
            time.sleep(pause * attempt)
    raise DataUnavailableError(f"{ticker}: download failed after {retries} attempts ({last_error})")


# --------------------------------------------------------------------------------------
# 2. Cleaning: statements -> one tidy table indexed by fiscal year end
# --------------------------------------------------------------------------------------
def _normalize_statement(df):
    """Rows = line items, columns = fiscal-year-end dates (ascending, timezone-free)."""
    if df is None or len(df) == 0:
        return pd.DataFrame()
    out = df.copy()
    out = out[~out.index.duplicated(keep="first")]
    out.columns = pd.DatetimeIndex(pd.to_datetime(out.columns)).tz_localize(None)
    out = out.apply(pd.to_numeric, errors="coerce")
    out = out.dropna(axis=1, how="all")
    return out.sort_index(axis=1)


def _pick(frame, names):
    for name in names:
        if name in frame.index and frame.loc[name].notna().any():
            return frame.loc[name].astype(float)
    return pd.Series(np.nan, index=frame.columns, dtype=float)


def fundamentals_table(raw):
    """One row per fiscal year end, one column per financial item (all in raw USD)."""
    frames = {
        "income": _normalize_statement(raw["income"]),
        "balance": _normalize_statement(raw["balance"]),
        "cashflow": _normalize_statement(raw["cashflow"]),
    }
    dates = sorted(set().union(*[set(f.columns) for f in frames.values() if len(f.columns)]))
    if not dates:
        raise DataUnavailableError(f"{raw.get('ticker', '?')}: statements contain no dates")
    table = pd.DataFrame(index=pd.DatetimeIndex(dates))
    for key, (statement, names) in SOURCES.items():
        frame = frames[statement]
        table[key] = _pick(frame, names).reindex(table.index) if len(frame) else np.nan
    # Free cash flow fallback: operating cash flow minus capital expenditure
    fallback = table["operating_cash_flow"] - table["capex"].abs()
    table["free_cash_flow"] = table["free_cash_flow"].fillna(fallback)
    table["interest_expense"] = table["interest_expense"].abs()
    table = table.dropna(subset=["revenue", "total_assets"], how="all").sort_index()
    return table


def close_series(prices):
    """Split-adjusted (not dividend-adjusted) closing prices with a timezone-free date index."""
    s = prices["Close"].dropna().astype(float)
    s.index = pd.DatetimeIndex(s.index).tz_localize(None).normalize()
    s = s[~s.index.duplicated(keep="last")].sort_index()
    return s[s > 0]


def split_factor_after(splits, date):
    """Product of stock-split ratios that happened AFTER `date`.

    Yahoo prices are adjusted to today's share units, while balance-sheet share counts are
    reported in the units of their own date. Multiplying the old share count by this factor
    puts both in the same units. (Verify with market_cap_consistency below.)
    """
    if splits is None or len(splits) == 0:
        return 1.0
    idx = pd.DatetimeIndex(splits.index).tz_localize(None)
    ratios = pd.Series(np.asarray(splits.values, dtype=float), index=idx)
    ratios = ratios[(ratios > 0) & (ratios.index > date)]
    return float(np.prod(ratios.values)) if len(ratios) else 1.0


def check_currency(raw):
    """Return (ok, message). We only support companies that report in the currency they trade in (USD)."""
    info = raw.get("info", {}) or {}
    reporting, trading = info.get("financialCurrency"), info.get("currency")
    if reporting and reporting != "USD":
        return False, f"reports in {reporting}, not USD"
    if trading and trading != "USD":
        return False, f"trades in {trading}, not USD"
    return True, "ok" if reporting else "currency not stated (assumed USD)"


def market_cap_consistency(raw, adjust_shares_for_splits=True):
    """Ratio of (latest close x latest reported shares) to Yahoo's current marketCap.

    A value far from 1 (say below 0.5 or above 2) usually means a share-count/split unit problem.
    Returns NaN when it cannot be computed.
    """
    try:
        fund = fundamentals_table(raw)
        close = close_series(raw["prices"])
        shares = fund["shares"].dropna()
        reported = raw["info"].get("marketCap")
        if shares.empty or not reported:
            return np.nan
        fye = shares.index[-1]
        factor = split_factor_after(raw.get("splits"), fye) if adjust_shares_for_splits else 1.0
        return float(close.iloc[-1] * shares.iloc[-1] * factor / reported)
    except Exception:
        return np.nan


def max_share_jump(raw, adjust_shares_for_splits=True):
    """Largest year-to-year change (as a multiple, >= 1) in split-adjusted reported share counts.

    A jump of ~10x means a stock split was missed or counted twice. Normal buybacks/issuance
    stay well below 3x, so values above 3 are treated as a data problem.
    """
    try:
        shares = fundamentals_table(raw)["shares"].dropna()
        if len(shares) < 2:
            return np.nan
        factors = np.array([split_factor_after(raw.get("splits"), d) if adjust_shares_for_splits else 1.0
                            for d in shares.index])
        adjusted = shares.values * factors
        ratios = adjusted[1:] / adjusted[:-1]
        return float(np.max(np.maximum(ratios, 1.0 / ratios)))
    except Exception:
        return np.nan


# --------------------------------------------------------------------------------------
# 3. Features (point-in-time: they use ONLY the rows of `hist`, i.e. the past)
# --------------------------------------------------------------------------------------
def _div(a, b):
    try:
        if not (np.isfinite(a) and np.isfinite(b)) or b == 0:
            return np.nan
    except TypeError:
        return np.nan
    return float(a) / float(b)


def _log(v):
    return float(np.log(v)) if np.isfinite(v) and v > 0 else np.nan


def _years(d0, d1):
    return (d1 - d0).days / 365.25


def _trapz(y, x):
    trapezoid = getattr(np, "trapezoid", None) or getattr(np, "trapz")
    return float(trapezoid(y, x))


def compute_features(hist):
    """Compute all model features from a fundamentals history that ENDS at the current fiscal year.

    `hist` is a slice of fundamentals_table(): rows are fiscal years up to (and including) the
    one being scored. Nothing after the last row can influence the result.
    """
    hist = hist.sort_index()
    n = len(hist)
    cur = hist.iloc[-1]
    f = {}

    rev, assets = cur["revenue"], cur["total_assets"]
    # ----- Core financial features -----
    f["log_revenue"] = _log(rev)
    f["log_total_assets"] = _log(assets)
    f["gross_margin"] = _div(cur["gross_profit"], rev)
    f["operating_margin"] = _div(cur["operating_income"], rev)
    f["net_margin"] = _div(cur["net_income"], rev)
    f["fcf_margin"] = _div(cur["free_cash_flow"], rev)
    f["roa"] = _div(cur["net_income"], assets)
    f["current_ratio"] = _div(cur["current_assets"], cur["current_liabilities"])
    f["cash_to_assets"] = _div(cur["cash"], assets)
    f["debt_to_assets"] = _div(cur["total_debt"], assets)
    f["liabilities_to_assets"] = _div(cur["total_liabilities"], assets)
    f["revenue_growth"] = np.nan
    if n >= 2:
        prev_rev = hist["revenue"].iloc[-2]
        f["revenue_growth"] = _div(rev - prev_rev, prev_rev)

    return f


def core_data_ok(cur):
    """A row is usable only if the basics needed for the target and main features exist."""
    return all(np.isfinite(cur[k]) and cur[k] > 0 for k in ("revenue", "total_assets", "shares"))


def build_company_rows(raw, ticker=None, report_lag_days=REPORT_LAG_DAYS,
                       adjust_shares_for_splits=True, fund=None):
    """Build one training row per fiscal year for a company.

    Row k describes the situation on `pred_date` = first trading day on/after
    (fiscal year end + report_lag_days). Features use fundamentals up to fiscal year k only.
    The target columns (market_cap, enterprise_value) use the price on pred_date.
    """
    ticker = ticker or raw.get("ticker", "?")
    fund = fundamentals_table(raw) if fund is None else fund
    close = close_series(raw["prices"])
    sector = (raw.get("info") or {}).get("sector") or "Unknown"
    splits = raw.get("splits")

    rows = []
    for k in range(MIN_HISTORY - 1, len(fund)):
        hist = fund.iloc[: k + 1]
        cur = hist.iloc[-1]
        fye = hist.index[-1]
        if not core_data_ok(cur):
            continue
        available = fye + pd.Timedelta(days=report_lag_days)
        pos = close.index.searchsorted(available)
        if pos >= len(close):
            continue                                   # statements not yet "public" in our price data
        pred_date = close.index[pos]
        if (pred_date - available).days > MAX_PRICE_GAP_DAYS:
            continue
        factor = split_factor_after(splits, fye) if adjust_shares_for_splits else 1.0
        shares_used = float(cur["shares"]) * factor
        price = float(close.iloc[pos])
        market_cap = price * shares_used
        debt, cash = cur["total_debt"], cur["cash"]
        enterprise_value = market_cap + debt - cash if np.isfinite(debt) and np.isfinite(cash) else np.nan

        row = {
            "ticker": ticker, "sector": sector,
            "fiscal_year_end": fye, "available_date": available, "pred_date": pred_date,
            "n_history": k + 1, "price": price, "shares_used": shares_used,
            "market_cap": market_cap, "enterprise_value": enterprise_value,
            "total_debt": debt, "cash": cash,
        }
        row.update(compute_features(hist))
        rows.append(row)
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------------------
# 4. Preprocessing (fitted ONLY on training data inside a scikit-learn Pipeline)
# --------------------------------------------------------------------------------------
class QuantileClipper(BaseEstimator, TransformerMixin):
    """Clip every column to the [lower, upper] quantiles learned from the training data."""

    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.low_ = np.nanquantile(X, self.lower, axis=0)
        self.high_ = np.nanquantile(X, self.upper, axis=0)
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return np.clip(X, self.low_, self.high_)

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)


def make_preprocessor(numeric_cols, categorical_cols=("sector",), scale=True):
    """Impute (median) -> clip outliers -> scale numeric columns; one-hot encode categories."""
    steps = [
        ("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("clip", QuantileClipper(0.01, 0.99)),
    ]
    if scale:
        steps.append(("scale", StandardScaler()))
    transformers = [("num", Pipeline(steps), list(numeric_cols))]
    if categorical_cols:
        transformers.append(("cat", OneHotEncoder(handle_unknown="ignore"), list(categorical_cols)))
    return ColumnTransformer(transformers, remainder="drop")

## 2.4 Company universe and data download

The list is a starting point (large US companies across sectors, several AI-related). Add or remove tickers as you like. Downloads are **cached** in the `data_cache/` folder, so re-running the cell does not download again. If Yahoo rate-limits you, wait a few minutes and re-run: only the missing companies are fetched.

Companies that fail any check are **excluded and listed with the reason**; the model and the final app only claim support for companies that load successfully.

In [ ]:
# 45. Configuration, company universe and data download
# (Downloaded data is cached in data_cache/. Delete that folder to download fresh data.)

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import valuation_pipeline as vp

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
CACHE_DIR = "data_cache"

UNIVERSE = [
    # Technology / AI-related
    "AAPL", "MSFT", "GOOGL", "AMZN", "META", "NVDA", "TSLA", "ORCL", "ADBE", "CRM",
    "CSCO", "INTC", "AMD", "QCOM", "TXN", "IBM", "NOW", "NFLX", "PLTR", "SNOW",
    # Consumer
    "NKE", "SBUX", "MCD", "HD", "COST", "WMT", "PG", "KO", "PEP", "DIS",
    # Healthcare
    "JNJ", "PFE", "MRK", "ABT", "LLY", "UNH",
    # Industrials, energy, telecom
    "CAT", "HON", "GE", "UPS", "XOM", "CVX", "VZ", "T",
]

raw_data = {}
load_log = []

for number, ticker in enumerate(UNIVERSE, start=1):
    was_cached = (Path(CACHE_DIR) / f"{ticker}.pkl").exists()
    try:
        raw = vp.fetch_raw(ticker, cache_dir=CACHE_DIR)
        ok, message = vp.check_currency(raw)
        if not ok:
            load_log.append({"ticker": ticker, "status": "excluded", "detail": message})
            continue
        ratio = vp.market_cap_consistency(raw)
        if np.isfinite(ratio) and not (0.5 <= ratio <= 2.0):
            load_log.append({"ticker": ticker, "status": "excluded",
                             "detail": f"share/price units look inconsistent (computed/Yahoo market cap = {ratio:.2f})"})
            continue
        jump = vp.max_share_jump(raw)
        if np.isfinite(jump) and jump > 3.0:
            load_log.append({"ticker": ticker, "status": "excluded",
                             "detail": f"share count jumps {jump:.1f}x between years (split missed or double-counted)"})
            continue
        raw_data[ticker] = raw
        load_log.append({"ticker": ticker, "status": "loaded",
                         "detail": f"market-cap check ratio = {ratio:.2f}" if np.isfinite(ratio) else "market-cap check unavailable"})
    except vp.DataUnavailableError as exc:
        load_log.append({"ticker": ticker, "status": "failed", "detail": str(exc)})
    if not was_cached:
        time.sleep(0.5)                       # be polite to the data provider

load_log = pd.DataFrame(load_log)
print(load_log["status"].value_counts().to_string())
display(load_log[load_log["status"] != "loaded"])

if len(raw_data) < 10:
    raise RuntimeError(
        f"Only {len(raw_data)} companies loaded successfully. This is usually a network problem or Yahoo "
        "rate-limiting. Wait a few minutes and run this cell again (companies already downloaded are "
        "cached and will not be downloaded again)."
    )
print(f"\nLoaded {len(raw_data)} companies.")
unknown_sector = [t for t, r in raw_data.items() if not (r.get("info") or {}).get("sector")]
no_check = [t for t, r in raw_data.items() if not np.isfinite(vp.market_cap_consistency(r))]
if unknown_sector or no_check:
    print(f"NOTE: {len(unknown_sector)} companies have no sector and {len(no_check)} could not be market-cap checked "
          "(Yahoo's 'info' endpoint is often rate-limited). To retry, delete the data_cache folder "
          "(or call vp.fetch_raw(ticker, refresh=True)) and run this cell again later.")

## 2.5 One row per company and fiscal year

Each row is "the situation on the prediction date". Features come from the statements that were public on that date. The target is the market capitalization on that date.

The cell also reports splits that happened after statements were published (a classic source of 10× unit errors), so you can sanity-check them.

In [ ]:
# 46. Build the modelling table (point-in-time features + target)

frames, build_log = [], []
for ticker, raw in raw_data.items():
    try:
        rows = vp.build_company_rows(raw, ticker=ticker)
    except vp.DataUnavailableError as exc:
        build_log.append({"ticker": ticker, "problem": str(exc)})
        continue
    if rows.empty:
        build_log.append({"ticker": ticker, "problem": "no usable fiscal years (missing core data or prices)"})
    else:
        frames.append(rows)

if not frames:
    raise RuntimeError("No usable rows were built. Check the download log above.")

dataset = (pd.concat(frames, ignore_index=True)
             .sort_values(["pred_date", "ticker"]).reset_index(drop=True))
dataset = dataset[(dataset["market_cap"] > 0) & np.isfinite(dataset["market_cap"])].reset_index(drop=True)
dataset["log_market_cap"] = np.log(dataset["market_cap"])

if build_log:
    print("Companies dropped while building rows:")
    display(pd.DataFrame(build_log))

coverage = pd.DataFrame({"fiscal years in statements": {t: len(vp.fundamentals_table(r)) for t, r in raw_data.items()}})
built_counts = dataset.groupby("ticker").size()
coverage["rows built"] = built_counts.reindex(coverage.index).fillna(0).astype(int)
print(f"Fiscal years found: {coverage['fiscal years in statements'].sum()}  ->  usable rows: {coverage['rows built'].sum()} "
      "(the first year of each company is only used as history; years with missing core data or prices are skipped)")

print(f"Rows: {len(dataset)}   Companies: {dataset['ticker'].nunique()}   "
      f"Prediction dates: {dataset['pred_date'].min().date()} to {dataset['pred_date'].max().date()}")
display(dataset.groupby("ticker").size().rename("rows").to_frame().T)

# Stock splits that happened after the statement date (these need share-count adjustment)
split_rows = []
for ticker, raw in raw_data.items():
    s = raw.get("splits")
    if s is not None and len(s):
        for date, ratio in s.items():
            if pd.Timestamp(date).tz_localize(None) >= dataset["fiscal_year_end"].min():
                split_rows.append({"ticker": ticker, "split date": pd.Timestamp(date).date(), "ratio": ratio})
print("\nStock splits inside the data window (share counts were adjusted for these):")
display(pd.DataFrame(split_rows) if split_rows else pd.DataFrame({"note": ["none found"]}))

display(dataset[["ticker", "fiscal_year_end", "available_date", "pred_date", "price",
                 "market_cap", "log_market_cap"]].head(8))

## 2.6 Data quality: missing values, outliers, scale

* **Missing values** are filled later with the *training-set median* (inside the pipeline). A large share of missing values in one feature is a warning sign.
* **Outliers**: we count values more than 5 robust standard deviations from the median (median/MAD). They are **clipped to the training 1st–99th percentile range** inside the pipeline instead of being deleted, because with so few rows we cannot afford to drop data.
* **Units**: all money values are raw USD. Ratios are unit-free, sizes use `log`.

In [ ]:
# 47. Data-quality report

feature_cols = vp.BASE_NUMERIC

quality = pd.DataFrame({
    "missing (%)": dataset[feature_cols].isna().mean() * 100,
    "min": dataset[feature_cols].min(),
    "median": dataset[feature_cols].median(),
    "max": dataset[feature_cols].max(),
})
def robust_outliers(series, limit=5):
    s = series.dropna()
    mad = np.median(np.abs(s - s.median()))
    if mad == 0:
        return 0
    return int((np.abs(s - s.median()) / (1.4826 * mad) > limit).sum())
quality["robust outliers (>5 MAD)"] = [robust_outliers(dataset[c]) for c in feature_cols]
display(quality.round(3))

high_missing = quality[quality["missing (%)"] > 30].index.tolist()
if high_missing:
    print("Features with more than 30% missing values:", high_missing)
    print("The pipeline fills missing values using medians learned from the training data.")
print("Sector counts:")
display(dataset.drop_duplicates("ticker")["sector"].value_counts().to_frame("companies").T)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(dataset["market_cap"] / 1e9, bins=30)
axes[0].set_title("Market cap (USD billions) - very skewed")
axes[1].hist(dataset["log_market_cap"], bins=30)
axes[1].set_title("log(market cap) - much more symmetric")
axes[2].scatter(dataset["log_revenue"], dataset["log_market_cap"], s=15)
axes[2].set_title("Size drives valuation: log revenue vs log market cap")
axes[2].set_xlabel("log revenue"); axes[2].set_ylabel("log market cap")
plt.tight_layout(); plt.show()

In [ ]:
# 48. Market capitalization vs enterprise value (they are different things)

ev_table = dataset[["ticker", "pred_date", "market_cap", "total_debt", "cash", "enterprise_value"]].copy()
ev_table["net debt"] = ev_table["total_debt"] - ev_table["cash"]
ev_table["EV / market cap"] = ev_table["enterprise_value"] / ev_table["market_cap"]
for col in ("market_cap", "total_debt", "cash", "enterprise_value", "net debt"):
    ev_table[col] = ev_table[col] / 1e9
ev_view = ev_table.rename(columns=lambda c: c + " (B USD)" if c in
                          ("market_cap", "total_debt", "cash", "enterprise_value", "net debt") else c)
number_cols = ev_view.select_dtypes("number").columns
ev_view[number_cols] = ev_view[number_cols].round(2)
display(ev_view.sample(8, random_state=RANDOM_SEED))

print("Distribution of EV / market cap across all rows:")
print(ev_table["EV / market cap"].describe().round(3).to_string())
print("\nThey differ because of net debt, so predicting one is not the same as predicting the other.")

## 2.7 Automated leakage checks

1. Every row's statements became "public" no later than its prediction date.
2. Price-based columns are not model inputs.
3. **Future-perturbation test:** we multiply the statements of *later* fiscal years by 100 and rebuild the table. If any earlier row's features change, the code is using the future. (The test uses real downloaded data only to verify the logic; it changes nothing permanently.)

In [ ]:
# 49. Leakage tests

# 1. Timing
assert (dataset["available_date"] <= dataset["pred_date"]).all(), "A prediction date is before the statements were public!"
assert ((dataset["pred_date"] - dataset["available_date"]).dt.days <= vp.MAX_PRICE_GAP_DAYS).all()
assert ((dataset["available_date"] - dataset["fiscal_year_end"]).dt.days == vp.REPORT_LAG_DAYS).all()
print("Check 1 passed: statements are always public before the prediction date.")

# 2. No price-based inputs
FORBIDDEN = {"price", "market_cap", "enterprise_value", "shares_used", "log_market_cap"}
assert FORBIDDEN.isdisjoint(vp.BASE_NUMERIC + vp.CATEGORICAL)
print("Check 2 passed: no price-based column is used as a feature.")

# 3. Future perturbation
tested = 0
for ticker, raw in raw_data.items():
    fund = vp.fundamentals_table(raw)
    if len(fund) < 3:
        continue
    cut = 1                                              # keep fiscal years 0..1 untouched
    perturbed = fund.copy()
    perturbed.iloc[cut + 1:] = perturbed.iloc[cut + 1:] * 100.0
    original_rows = vp.build_company_rows(raw, ticker=ticker, fund=fund)
    changed_rows = vp.build_company_rows(raw, ticker=ticker, fund=perturbed)
    keep = original_rows["fiscal_year_end"] <= fund.index[cut]
    if keep.sum() == 0:
        continue
    cols = vp.BASE_NUMERIC
    a = original_rows.loc[keep, cols].to_numpy(float)
    b = changed_rows[changed_rows["fiscal_year_end"] <= fund.index[cut]][cols].to_numpy(float)
    assert a.shape == b.shape and np.allclose(a, b, equal_nan=True), f"{ticker}: features changed when only FUTURE data changed -> leakage!"
    tested += 1
    if tested >= 10:
        break
print(f"Check 3 passed: earlier rows are unchanged when future statements are altered ({tested} companies tested).")

## 2.8 Chronological train / validation / test split

Rows are ordered by **prediction date**: the earliest 60% are the training set, the next 20% the validation set (used to choose between models) and the latest 20% the test set (looked at only for the final report). No random shuffling — that would let the model train on the future.

**Important limitation:** the same company can appear in training (older years) and test (newer years). So the test measures "predict a *known* company's next valuation", which is easier than "value a *brand-new* company". We report how much the test set overlaps with training companies.

In [ ]:
# 50. Chronological split

TARGET = "log_market_cap"
FEATURES_BASE = list(vp.BASE_NUMERIC)
CATEGORICAL = list(vp.CATEGORICAL)

dataset = dataset.sort_values("pred_date").reset_index(drop=True)
cut_train = dataset["pred_date"].quantile(0.60)
cut_val = dataset["pred_date"].quantile(0.80)

train = dataset[dataset["pred_date"] <= cut_train].reset_index(drop=True)
val = dataset[(dataset["pred_date"] > cut_train) & (dataset["pred_date"] <= cut_val)].reset_index(drop=True)
test = dataset[dataset["pred_date"] > cut_val].reset_index(drop=True)

for name, part in (("train", train), ("validation", val), ("test", test)):
    if len(part) < 8:
        raise RuntimeError(f"The {name} set has only {len(part)} rows. Add more companies to UNIVERSE or "
                           "check the download log - the dataset is too small to split.")

assert train["pred_date"].max() < val["pred_date"].min() <= val["pred_date"].max() < test["pred_date"].min()

split_summary = pd.DataFrame({
    "rows": [len(train), len(val), len(test)],
    "from": [train["pred_date"].min().date(), val["pred_date"].min().date(), test["pred_date"].min().date()],
    "to": [train["pred_date"].max().date(), val["pred_date"].max().date(), test["pred_date"].max().date()],
    "companies": [train["ticker"].nunique(), val["ticker"].nunique(), test["ticker"].nunique()],
}, index=["train", "validation", "test"])
display(split_summary)

seen = test["ticker"].isin(train["ticker"]).mean()
print(f"{seen:.0%} of test rows belong to companies that also appear in the training set.")
if len(dataset) < 100:
    print("NOTE: fewer than 100 rows in total - treat every metric below as noisy and indicative only.")

---

# PART 3 — Machine Learning

## 3.1 Plan and honesty rules

| Step | What we do |
|---|---|
| Baselines | (a) always predict the training average; (b) "revenue multiple": value = revenue × the typical training market-cap/revenue ratio. A real model must beat these to be worth anything. |
| Models | **Linear Regression**, **Ridge** (justified: features such as `debt_to_assets` and `liabilities_to_assets` are strongly correlated and the dataset is small, so a penalty on large coefficients can help) and **Random Forest**. No other models: with so few rows, more complex models cannot be justified. |
| Tuning | Hyperparameters are chosen with **expanding-window cross-validation** on the training set (always train on the past, validate on the next block of dates). |
| Selection | The validation set decides which model performs best. |
| Final report | Models are refitted on train + validation and evaluated **once** on the test set. |

**Metrics** (all on the log scale of market cap, because that is what the models predict): **MAE** (average absolute error), **RMSE** (penalizes large errors) and **R²** (share of variance explained). Because 0.1 on a log scale ≈ a 10% error, we also report the **median absolute percentage error** in dollar terms.

Note: R² on log market cap looks flattering because company *size* alone explains much of the variation. That is why the baselines matter.

In [ ]:
# 51. Evaluation helpers and the two baselines

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate(y_true_log, y_pred_log):
    y_true_log, y_pred_log = np.asarray(y_true_log, float), np.asarray(y_pred_log, float)
    error = y_pred_log - y_true_log
    return {
        "MAE (log)": mean_absolute_error(y_true_log, y_pred_log),
        "RMSE (log)": float(np.sqrt(mean_squared_error(y_true_log, y_pred_log))),
        "R2": r2_score(y_true_log, y_pred_log),
        "median abs % error": float(100 * np.median(np.abs(np.expm1(error)))),
    }

def baseline_mean(train_df, df):
    """Predict the training-set average log market cap for every company."""
    return np.full(len(df), train_df[TARGET].mean())

def baseline_revenue_multiple(train_df, df):
    """Predict revenue x (typical training market-cap / revenue ratio)."""
    typical = np.median(train_df[TARGET] - train_df["log_revenue"])
    return df["log_revenue"].fillna(train_df["log_revenue"].median()).to_numpy() + typical

BASELINES = {"Baseline: training mean": baseline_mean,
             "Baseline: revenue multiple": baseline_revenue_multiple}

baseline_val = pd.DataFrame({name: evaluate(val[TARGET], fn(train, val)) for name, fn in BASELINES.items()}).T
print("Baselines on the VALIDATION set (fitted on the training set):")
display(baseline_val.round(3))

## 3.2 Models, hyperparameter grids and time-aware cross-validation

`expanding_window_folds` splits the training rows into 4 chronological blocks and creates folds such as *train on block 1 → validate on block 2*, *train on blocks 1-2 → validate on block 3*, and so on. This respects time order, unlike ordinary shuffled k-fold cross-validation.

Preprocessing (median imputation → clip outliers → standardize, plus one-hot encoding of `sector`) is part of the pipeline, so it is re-fitted inside every fold on that fold's training rows only.

In [ ]:
# 52. Pipelines, grids and chronological cross-validation

from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline

def make_pipeline(model_name, numeric_cols):
    if model_name == "Linear Regression":
        estimator, scale = LinearRegression(), True
    elif model_name == "Ridge":
        estimator, scale = Ridge(), True
    elif model_name == "Random Forest":
        estimator, scale = RandomForestRegressor(n_estimators=300, random_state=RANDOM_SEED, n_jobs=-1), False
    else:
        raise ValueError(f"Unknown model: {model_name}")
    return Pipeline([("prep", vp.make_preprocessor(numeric_cols, CATEGORICAL, scale=scale)),
                     ("model", estimator)])

PARAM_GRIDS = {
    "Linear Regression": {},
    "Ridge": {"model__alpha": [0.1, 1.0, 10.0, 100.0]},
    "Random Forest": {"model__max_depth": [3, 5, None],
                      "model__min_samples_leaf": [1, 3, 5],
                      "model__max_features": [0.5, 1.0]},
}

def expanding_window_folds(dates, n_blocks=4, min_train=8, min_valid=3):
    """Chronological folds: train on earlier date-blocks, validate on the next block."""
    dates = pd.Series(pd.to_datetime(dates)).reset_index(drop=True)
    blocks = np.array_split(np.sort(dates.unique()), n_blocks)
    folds = []
    for i in range(1, len(blocks)):
        train_mask = dates.isin(np.concatenate(blocks[:i]))
        valid_mask = dates.isin(blocks[i])
        if train_mask.sum() >= min_train and valid_mask.sum() >= min_valid:
            folds.append((np.where(train_mask)[0], np.where(valid_mask)[0]))
    if not folds:
        raise RuntimeError("Not enough rows to build chronological cross-validation folds.")
    return folds

demo_folds = expanding_window_folds(train["pred_date"])
print("Cross-validation folds (rows in train part -> rows in validation part):")
for i, (a, b) in enumerate(demo_folds, 1):
    print(f"  fold {i}: {len(a)} -> {len(b)}")

## 3.3 Training and tuning the candidate models

Each of the three models uses the same 12 core financial features. Linear Regression has no hyperparameters to tune; Ridge and Random Forest use chronological cross-validation to select their settings.

In [ ]:
# 53. Tune each candidate with chronological cross-validation, then score on the validation set

FEATURE_SETS = {"core financial": FEATURES_BASE}
MODEL_NAMES = ["Linear Regression", "Ridge", "Random Forest"]

candidates = {}
folds = expanding_window_folds(train["pred_date"])
X_train_all = {k: train[cols + CATEGORICAL] for k, cols in FEATURE_SETS.items()}
y_train = train[TARGET]

for feature_set, numeric_cols in FEATURE_SETS.items():
    for model_name in MODEL_NAMES:
        pipe = make_pipeline(model_name, numeric_cols)
        grid = PARAM_GRIDS[model_name]
        X_tr = X_train_all[feature_set]
        if grid:
            search = GridSearchCV(pipe, grid, cv=folds, scoring="neg_root_mean_squared_error", n_jobs=1)
            search.fit(X_tr, y_train)
            fitted, cv_rmse, best_params = search.best_estimator_, -search.best_score_, search.best_params_
        else:
            scores = cross_val_score(pipe, X_tr, y_train, cv=folds, scoring="neg_root_mean_squared_error")
            fitted, cv_rmse, best_params = pipe.fit(X_tr, y_train), -scores.mean(), {}
        preds = fitted.predict(val[numeric_cols + CATEGORICAL])
        candidates[(model_name, feature_set)] = {
            "pipeline": fitted, "numeric_cols": numeric_cols, "best_params": best_params,
            "cv_rmse": cv_rmse, **{f"val {k}": v for k, v in evaluate(val[TARGET], preds).items()},
        }
        print(f"trained: {model_name:<18} | {feature_set:<16} | CV RMSE {cv_rmse:.3f} | params {best_params}")

dev_table = pd.DataFrame(
    {key: {k: v for k, v in info.items() if k in ("cv_rmse", "val MAE (log)", "val RMSE (log)", "val R2", "val median abs % error")}
     for key, info in candidates.items()}).T
dev_table.index.names = ["model", "features"]
baseline_rows = baseline_val.rename(columns={
    "MAE (log)": "val MAE (log)", "RMSE (log)": "val RMSE (log)", "R2": "val R2",
    "median abs % error": "val median abs % error"})
baseline_rows.index = pd.MultiIndex.from_tuples([(n, "-") for n in baseline_val.index], names=["model", "features"])
dev_table = pd.concat([dev_table, baseline_rows])
print("\nDevelopment results (cross-validation on train, then validation set):")
display(dev_table.sort_values("val RMSE (log)").round(3))

best_key = min(candidates, key=lambda k: candidates[k]["val RMSE (log)"])
print(f"\nBest candidate by validation RMSE: {best_key[0]} with '{best_key[1]}' features")

## 3.4 Final evaluation on the untouched test set

Every candidate is refitted on **train + validation** (using the hyperparameters found above) and scored on the test set **once**. We do not go back and re-tune after seeing these numbers. The deployment model is the one chosen by *validation* performance, not the one that happens to look best on the test set.

In [ ]:
# 54. Refit on train + validation, evaluate on the test set once

train_val = pd.concat([train, val], ignore_index=True)
final_models = {}
test_rows = {}

for (model_name, feature_set), info in candidates.items():
    numeric_cols = info["numeric_cols"]
    pipe = clone(make_pipeline(model_name, numeric_cols)).set_params(**info["best_params"])
    pipe.fit(train_val[numeric_cols + CATEGORICAL], train_val[TARGET])
    preds = pipe.predict(test[numeric_cols + CATEGORICAL])
    final_models[(model_name, feature_set)] = pipe
    test_rows[(model_name, feature_set)] = {**evaluate(test[TARGET], preds), "pred": preds}

test_table = pd.DataFrame({k: {m: v for m, v in row.items() if m != "pred"} for k, row in test_rows.items()}).T
test_table.index.names = ["model", "features"]
baseline_test = pd.DataFrame({name: evaluate(test[TARGET], fn(train_val, test)) for name, fn in BASELINES.items()}).T
baseline_test.index = pd.MultiIndex.from_tuples([(n, "-") for n in baseline_test.index], names=["model", "features"])
test_table = pd.concat([test_table, baseline_test])

print("TEST SET results (models refitted on train + validation):")
display(test_table.sort_values("RMSE (log)").round(3))
print(f"Deployment candidate (chosen on validation, not on test): {best_key}")

## 3.6 Explainability — what drives the predictions?

* **Permutation importance** (works for any model): shuffle one input column in the *validation* set and measure how much the error grows. A big increase means the model relies on that column. It is computed with models fitted on the training set only.
* **Random Forest impurity importance:** how much each feature reduced the squared error inside the trees (can be biased toward continuous features).
* **Ridge coefficients:** on standardized features, so sizes are comparable; the sign shows direction.

Importance describes *what the model uses*, not *what causes valuation*.

In [ ]:
# 56. Permutation importance, Random Forest importance and Ridge coefficients

from sklearn.inspection import permutation_importance

def shorten(names):
    return [n.replace("num__", "").replace("cat__", "") for n in names]

rf_key = ("Random Forest", best_key[1])           # explain the RF that uses the same feature set as the best candidate
rf_pipe = candidates[rf_key]["pipeline"]            # trained on the training set only
cols = candidates[rf_key]["numeric_cols"] + CATEGORICAL

perm = permutation_importance(rf_pipe, val[cols], val[TARGET], n_repeats=30, random_state=RANDOM_SEED,
                              scoring="neg_root_mean_squared_error")
perm_table = pd.DataFrame({"feature": cols, "RMSE increase when shuffled": perm.importances_mean,
                           "std": perm.importances_std}).sort_values("RMSE increase when shuffled", ascending=False)

feature_names = shorten(rf_pipe.named_steps["prep"].get_feature_names_out())
impurity = pd.Series(rf_pipe.named_steps["model"].feature_importances_, index=feature_names).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
top = perm_table.head(12).iloc[::-1]
axes[0].barh(top["feature"], top["RMSE increase when shuffled"], xerr=top["std"])
axes[0].set_title(f"Permutation importance (validation) - Random Forest, {rf_key[1]}")
axes[0].set_xlabel("increase in RMSE (log) when the column is shuffled")
top_i = impurity.head(12).iloc[::-1]
axes[1].barh(top_i.index, top_i.values)
axes[1].set_title("Random Forest impurity importance")
plt.tight_layout(); plt.show()

# Ridge coefficients (standardized inputs)
ridge_key = ("Ridge", best_key[1])
ridge_pipe = candidates[ridge_key]["pipeline"]
ridge_names = shorten(ridge_pipe.named_steps["prep"].get_feature_names_out())
ridge_coef = pd.Series(ridge_pipe.named_steps["model"].coef_, index=ridge_names).sort_values(key=np.abs, ascending=False)
print(f"Largest Ridge coefficients (standardized features; alpha = {candidates[ridge_key]['best_params']}):")
display(ridge_coef.head(12).round(3).to_frame("coefficient").T)



## 3.7 Error analysis of the chosen model on the test set

Where does the model fail? A scatter plot of predicted vs actual values, and the largest individual errors.

In [ ]:
# 57. Predicted vs actual and the worst errors (deployment candidate, test set)

best_pred = test_rows[best_key]["pred"]
analysis = test[["ticker", "sector", "pred_date", "market_cap"]].copy()
analysis["predicted market cap"] = np.exp(best_pred)
analysis["error (%)"] = 100 * (analysis["predicted market cap"] / analysis["market_cap"] - 1)
analysis["abs error (%)"] = analysis["error (%)"].abs()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
lims = [test[TARGET].min() - 0.3, test[TARGET].max() + 0.3]
axes[0].scatter(test[TARGET], best_pred, s=25)
axes[0].plot(lims, lims, "r--", label="perfect prediction")
axes[0].set_xlabel("actual log market cap"); axes[0].set_ylabel("predicted log market cap")
axes[0].set_title(f"{best_key[0]} ({best_key[1]})"); axes[0].legend(); axes[0].grid(True)
sector_err = analysis.groupby("sector")["abs error (%)"].median().sort_values()
axes[1].barh(sector_err.index, sector_err.values)
axes[1].set_title("Median absolute % error by sector (test set)"); axes[1].set_xlabel("%")
plt.tight_layout(); plt.show()

worst = (analysis.sort_values("abs error (%)", ascending=False).head(8)
         .assign(**{"market cap (B USD)": lambda d: (d["market_cap"] / 1e9).round(1),
                    "predicted (B USD)": lambda d: (d["predicted market cap"] / 1e9).round(1),
                    "error (%)": lambda d: d["error (%)"].round(1)})
         [["ticker", "sector", "pred_date", "market cap (B USD)", "predicted (B USD)", "error (%)"]])
display(worst)

## 3.8 Gradient descent on the real data (linear regression from scratch)

Linear regression minimizes the **mean squared error** $L(\mathbf w)=\frac1n\lVert X\mathbf w-\mathbf y\rVert^2$. This loss is differentiable, so we can use the calculus from Part 1:

* Gradient: $\nabla L=\dfrac{2}{n}X^{\top}(X\mathbf w-\mathbf y)$
* Hessian: $H=\dfrac{2}{n}X^{\top}X$ — always positive semidefinite, so $L$ is **convex** and has no bad local minima.
* Gradient descent is stable when the learning rate is below $2/\lambda_{\max}(H)$.

We run it on the (standardized) **base** features of the training set and compare with the exact least-squares solution. *Random Forests are not trained this way* — they grow trees by greedily choosing split thresholds, which involves no gradients. (Gradient *boosting* is a different model that does use gradients, but we do not use it here.)

In [ ]:
# 58. Gradient descent for linear regression on the real training data

prep_gd = vp.make_preprocessor(FEATURES_BASE, categorical_cols=(), scale=True).fit(train[FEATURES_BASE])
X_gd = prep_gd.transform(train[FEATURES_BASE])
y_gd = train[TARGET].to_numpy()
X_aug = np.column_stack([np.ones(len(X_gd)), X_gd])          # add an intercept column
n_obs = len(y_gd)

loss = lambda w: float(np.mean((X_aug @ w - y_gd) ** 2))
gradient = lambda w: (2.0 / n_obs) * X_aug.T @ (X_aug @ w - y_gd)

hessian = (2.0 / n_obs) * X_aug.T @ X_aug
eigs = np.linalg.eigvalsh(hessian)
lam_max, lam_min = eigs.max(), max(eigs.min(), 1e-12)
print(f"Hessian eigenvalues: min = {eigs.min():.4g}, max = {lam_max:.4g}  (all >= 0 -> convex loss)")
print(f"Condition number = {lam_max / lam_min:,.0f}  (large values make gradient descent slow)")
print(f"Stable learning rates: below 2 / lambda_max = {2 / lam_max:.4f}")

learning_rate = 1.0 / lam_max
w = np.zeros(X_aug.shape[1])
losses = [loss(w)]
for step in range(100_000):
    g = gradient(w)
    if np.linalg.norm(g) < 1e-9:
        break
    w = w - learning_rate * g
    losses.append(loss(w))
converged = np.linalg.norm(gradient(w)) < 1e-6
print(f"\nStopped after {step + 1} iterations; final loss {losses[-1]:.5f}; gradient norm {np.linalg.norm(gradient(w)):.2e}")

w_exact, *_ = np.linalg.lstsq(X_aug, y_gd, rcond=None)
sk_model = LinearRegression().fit(X_gd, y_gd)
print(f"Max |GD prediction - exact least-squares prediction| on training rows: {np.max(np.abs(X_aug @ w - X_aug @ w_exact)):.2e}")
print(f"Max |GD prediction - sklearn LinearRegression prediction|:            {np.max(np.abs(X_aug @ w - sk_model.predict(X_gd))):.2e}")
if not converged:
    print("Gradient descent had not fully converged (ill-conditioned features); the exact solution is more reliable here.")

# What happens with a learning rate that is too large?
w_bad = np.zeros(X_aug.shape[1]); bad_losses = [loss(w_bad)]
for _ in range(60):
    w_bad = w_bad - (2.2 / lam_max) * gradient(w_bad)
    bad_losses.append(loss(w_bad))
print(f"With learning rate = 2.2 / lambda_max the loss goes from {bad_losses[0]:.2f} to {bad_losses[-1]:.3g}  (diverges)")

plt.figure(figsize=(9, 4))
plt.semilogy(losses)
plt.title("Gradient descent: training MSE per iteration (log scale)")
plt.xlabel("iteration"); plt.ylabel("MSE (log market cap)"); plt.grid(True, which="both", alpha=0.3)
plt.show()

## 3.9 Automatic honesty report

The cell below turns the numbers above into plain statements, so the conclusions in your report come from the data and are not typed in by hand. Re-run it after any change.

In [ ]:
# 59. Automatic summary of what the results do and do not show

best_test = test_rows[best_key]
best_vs_baseline = baseline_test.loc[("Baseline: revenue multiple", "-")]
print(f"Dataset: {len(dataset)} rows, {dataset['ticker'].nunique()} companies; "
      f"train/val/test = {len(train)}/{len(val)}/{len(test)} rows.")
print(f"Chosen model (validation): {best_key[0]} with '{best_key[1]}' features.")
print(f"  test MAE (log) = {best_test['MAE (log)']:.3f}, RMSE (log) = {best_test['RMSE (log)']:.3f}, "
      f"R2 = {best_test['R2']:.3f}, median abs % error = {best_test['median abs % error']:.1f}%")
print(f"  revenue-multiple baseline: MAE (log) = {best_vs_baseline['MAE (log)']:.3f}, "
      f"R2 = {best_vs_baseline['R2']:.3f}, median abs % error = {best_vs_baseline['median abs % error']:.1f}%")

if best_test["RMSE (log)"] < best_vs_baseline["RMSE (log)"]:
    print("  -> The chosen model beats the revenue-multiple baseline on the test set.")
else:
    print("  -> The chosen model does NOT beat the simple revenue-multiple baseline on the test set. "
          "Do not claim it adds value; more data or better features are needed.")

beaten = [k for k in test_rows if test_rows[k]["RMSE (log)"] >= baseline_test.loc[("Baseline: training mean", "-")]["RMSE (log)"]]
if beaten:
    print("  -> Candidates no better than predicting the training mean:", beaten)

print("\nLimitations that apply to every number above:")
for item in (
    f"Small sample ({len(dataset)} rows) -> wide uncertainty; the test set has only {len(test)} rows.",
    "No market-wide variable: if the whole market re-prices, predictions drift because the model learned average multiples of the training period.",
    "Test companies mostly also appear in training, so the test is easier than valuing a brand-new company.",
    "Survivorship bias (today's large companies) and a single market regime.",
    "Yahoo figures are restated, and filing dates are assumed (90-day lag).",
):
    print("  -", item)

---

# PART 4 — Company Health Analysis (rule-based, separate from the ML model)

The health score answers a different question from the valuation model:

| | Valuation model (Part 3) | Health score (this part) |
|---|---|---|
| Question | What market cap do companies like this usually have? | Is the company's financial position sound? |
| Method | Learned from data | **Fixed, documented rules** |
| Uses prices? | Target only | **Never** |
| Validated against outcomes? | Yes, on a held-out test set | **No** — it is a transparent heuristic |

## How the score works

1. Indicators are grouped into five categories: **profitability, liquidity, leverage, cash flow, growth**.
2. Each indicator is converted to 0–100 points by a *linear rule*: the "0 points" level and the "100 points" level are written in the table below; values in between are interpolated and values beyond are capped.
3. A category score is the average of its available indicators. Missing indicators are **ignored**, not treated as zero.
4. The overall score is the weighted average of the available categories (weights below, re-normalized if a category is missing). **At least 3 of 5 categories are required**, otherwise the result is "Insufficient data".
5. Grades: **≥ 75 Strong, ≥ 55 Moderate, ≥ 35 Weak, < 35 Distressed.**

## Limitations (important)

* Thresholds are general rules of thumb, **not sector-specific**. For example, utilities legitimately carry more debt than software firms, and retailers run low current ratios.
* One year of statements can be distorted by one-off items. Unusual accounting (negative equity from buybacks, as at some large consumer brands) is penalized by the leverage rule even when the business is strong.
* The weights are judgement calls and have not been calibrated against bankruptcies or credit ratings.
* Banks and insurers are outside the scope of these indicators.

The scoring code is saved as `company_health.py` so that the API can reuse it.

In [ ]:
%%writefile company_health.py
"""
company_health.py
=================
A transparent, rule-based financial HEALTH SCORE (0-100).

This is deliberately separate from the machine-learning valuation model:
* it is NOT trained on data and NOT validated against outcomes such as bankruptcies;
* every rule, threshold and weight is written down in HEALTH_RULES below;
* it only uses the company's own financial statements (no stock price, no ML output).
"""

import numpy as np
import pandas as pd

import valuation_pipeline as vp

# poor  -> indicator value that scores 0 points
# good  -> indicator value that scores 100 points (linear in between, clipped to 0..100)
# "poor" may be larger than "good" for indicators where lower is better (e.g. debt).
HEALTH_RULES = {
    "Profitability": {"weight": 0.25, "indicators": {
        "operating_margin": {"poor": 0.00, "good": 0.25, "text": "Operating income / revenue"},
        "net_margin":       {"poor": 0.00, "good": 0.20, "text": "Net income / revenue"},
        "roa":              {"poor": 0.00, "good": 0.10, "text": "Net income / total assets"},
    }},
    "Liquidity": {"weight": 0.15, "indicators": {
        "current_ratio":    {"poor": 0.80, "good": 2.00, "text": "Current assets / current liabilities"},
        "cash_ratio":       {"poor": 0.10, "good": 1.00, "text": "Cash & short-term investments / current liabilities"},
    }},
    "Leverage": {"weight": 0.20, "indicators": {
        "debt_to_equity":   {"poor": 3.00, "good": 0.50, "text": "Total debt / shareholders' equity (lower is better; negative equity scores 0)"},
        "interest_coverage": {"poor": 1.00, "good": 10.0, "text": "Operating income / interest expense"},
    }},
    "Cash flow": {"weight": 0.25, "indicators": {
        "fcf_margin":       {"poor": 0.00, "good": 0.20, "text": "Free cash flow / revenue"},
        "ocf_to_net_income": {"poor": 0.60, "good": 1.20, "text": "Operating cash flow / net income (only when net income > 0)"},
    }},
    "Growth": {"weight": 0.15, "indicators": {
        "revenue_growth":   {"poor": -0.05, "good": 0.15, "text": "Latest year revenue growth"},
        "revenue_cagr":     {"poor": -0.05, "good": 0.15, "text": "Revenue growth per year over up to 3 years"},
    }},
}

GRADE_BANDS = [(75, "Strong"), (55, "Moderate"), (35, "Weak"), (0, "Distressed")]
MIN_CATEGORIES = 3        # need at least this many categories with data, else no score


def rules_table():
    """The scoring rules as a table (useful to show in the notebook or the app)."""
    rows = []
    for category, spec in HEALTH_RULES.items():
        for name, rule in spec["indicators"].items():
            rows.append({"category": category, "category weight": spec["weight"], "indicator": name,
                         "0 points at": rule["poor"], "100 points at": rule["good"],
                         "definition": rule["text"]})
    return pd.DataFrame(rows)


def health_inputs(hist):
    """All indicator values from a fundamentals history that ends at the latest fiscal year."""
    hist = hist.sort_index()
    base = vp.compute_features(hist)
    cur = hist.iloc[-1]
    n = len(hist)
    out = {k: base.get(k, np.nan) for k in ("operating_margin", "net_margin", "roa", "current_ratio",
                                           "fcf_margin", "revenue_growth")}
    out["cash_ratio"] = vp._div(cur["cash"], cur["current_liabilities"])

    equity, debt = cur["equity"], cur["total_debt"]
    if np.isfinite(equity) and np.isfinite(debt):
        out["debt_to_equity"] = debt / equity if equity > 0 else (np.inf if debt > 0 else np.nan)
    else:
        out["debt_to_equity"] = np.nan

    interest = cur["interest_expense"]
    out["interest_coverage"] = (vp._div(cur["operating_income"], interest)
                                if np.isfinite(interest) and interest > 0 else np.nan)

    ni = cur["net_income"]
    out["ocf_to_net_income"] = (vp._div(cur["operating_cash_flow"], ni)
                                if np.isfinite(ni) and ni > 0 else np.nan)

    out["revenue_cagr"] = np.nan
    if n >= 2:
        m = min(3, n - 1)
        start = hist["revenue"].iloc[-1 - m]
        years = vp._years(hist.index[-1 - m], hist.index[-1])
        if np.isfinite(start) and start > 0 and np.isfinite(cur["revenue"]) and cur["revenue"] > 0 and years > 0:
            out["revenue_cagr"] = (cur["revenue"] / start) ** (1 / years) - 1
    return out


def score_indicator(value, poor, good):
    """Linear 0-100 score. Returns None when the value is missing."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    ratio = (value - poor) / (good - poor)
    return float(np.clip(ratio, 0.0, 1.0) * 100.0)


def grade_for(score):
    for threshold, label in GRADE_BANDS:
        if score >= threshold:
            return label
    return "Distressed"


def score_company(inputs):
    """Turn indicator values into category scores, an overall score and a grade."""
    indicator_rows, category_scores, weights = [], {}, {}
    for category, spec in HEALTH_RULES.items():
        scores = []
        for name, rule in spec["indicators"].items():
            value = inputs.get(name, np.nan)
            s = score_indicator(value, rule["poor"], rule["good"])
            indicator_rows.append({"category": category, "indicator": name,
                                   "value": value, "score": s})
            if s is not None:
                scores.append(s)
        if scores:
            category_scores[category] = float(np.mean(scores))
            weights[category] = spec["weight"]

    result = {"category_scores": category_scores,
              "indicators": pd.DataFrame(indicator_rows),
              "categories_used": len(category_scores)}
    if len(category_scores) < MIN_CATEGORIES:
        result.update(overall_score=None, grade="Insufficient data")
        return result
    total_weight = sum(weights.values())
    overall = sum(category_scores[c] * weights[c] for c in category_scores) / total_weight
    result.update(overall_score=float(overall), grade=grade_for(overall))
    return result


def health_report(hist):
    """Convenience wrapper: fundamentals history -> scored report."""
    report = score_company(health_inputs(hist))
    report["as_of_fiscal_year_end"] = pd.Timestamp(hist.index.max())
    return report

In [ ]:
# 60. Show the scoring rules and run unit tests on the scoring function

import company_health as ch

display(ch.rules_table())

# Unit tests with INVENTED test inputs (not real companies) to confirm the rules behave sensibly.
def make_test_history(op_margin, net_margin, current_ratio, debt, equity, fcf_margin, growth):
    revenues = [100.0, 100.0 * (1 + growth), 100.0 * (1 + growth) ** 2, 100.0 * (1 + growth) ** 3]
    dates = pd.to_datetime(["2022-12-31", "2023-12-31", "2024-12-31", "2025-12-31"])
    rows = []
    for rev in revenues:
        rows.append({"revenue": rev, "gross_profit": rev * 0.5, "operating_income": rev * op_margin,
                     "net_income": rev * net_margin, "interest_expense": 2.0, "total_assets": 200.0,
                     "total_liabilities": 100.0, "equity": equity, "current_assets": 60.0 * current_ratio,
                     "current_liabilities": 60.0, "cash": 30.0, "total_debt": debt, "shares": 10.0,
                     "operating_cash_flow": rev * (fcf_margin + 0.03), "free_cash_flow": rev * fcf_margin,
                     "capex": -rev * 0.03})
    return pd.DataFrame(rows, index=dates)

strong = ch.health_report(make_test_history(0.30, 0.22, 2.2, 20.0, 100.0, 0.22, 0.12))
average = ch.health_report(make_test_history(0.12, 0.08, 1.3, 60.0, 80.0, 0.08, 0.04))
weak = ch.health_report(make_test_history(-0.05, -0.08, 0.6, 150.0, 20.0, -0.04, -0.06))
print(f"strong: {strong['overall_score']:.1f} ({strong['grade']}) | average: {average['overall_score']:.1f} "
      f"({average['grade']}) | weak: {weak['overall_score']:.1f} ({weak['grade']})")
assert strong["overall_score"] > average["overall_score"] > weak["overall_score"], "Score ordering is wrong!"
assert strong["grade"] == "Strong" and weak["grade"] in ("Weak", "Distressed")

# More profit must never lower the score (monotonic), and missing data must not crash the code.
low = ch.score_company({"operating_margin": 0.05, "net_margin": 0.03, "roa": 0.02, "current_ratio": 1.5,
                        "cash_ratio": 0.5, "debt_to_equity": 1.0, "interest_coverage": 5.0,
                        "fcf_margin": 0.05, "ocf_to_net_income": 1.0, "revenue_growth": 0.05, "revenue_cagr": 0.05})
high = ch.score_company({"operating_margin": 0.20, "net_margin": 0.03, "roa": 0.02, "current_ratio": 1.5,
                         "cash_ratio": 0.5, "debt_to_equity": 1.0, "interest_coverage": 5.0,
                         "fcf_margin": 0.05, "ocf_to_net_income": 1.0, "revenue_growth": 0.05, "revenue_cagr": 0.05})
assert high["overall_score"] >= low["overall_score"]
sparse = ch.score_company({"operating_margin": 0.1})
assert sparse["grade"] == "Insufficient data" and sparse["overall_score"] is None
print("Unit tests passed: ordering, monotonicity and missing-data handling behave as documented.")

## Health scores for the companies in our dataset

For each company we score its **latest available fiscal year**. Scores are *not* used as features in the ML model.

In [ ]:
# 61. Health score for every loaded company (latest fiscal year)

health_rows, health_reports = [], {}
for ticker, raw in raw_data.items():
    try:
        report = ch.health_report(vp.fundamentals_table(raw))
    except Exception as exc:
        health_rows.append({"ticker": ticker, "grade": f"error: {exc}"})
        continue
    health_reports[ticker] = report
    health_rows.append({"ticker": ticker, "as of": report["as_of_fiscal_year_end"].date(),
                        "overall": report["overall_score"], "grade": report["grade"],
                        "categories used": report["categories_used"], **report["category_scores"]})

health_table = pd.DataFrame(health_rows).set_index("ticker").sort_values("overall", ascending=False)
display(health_table.round(1))
print(health_table["grade"].value_counts().to_string())

scored = health_table.dropna(subset=["overall"])
if len(scored):
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(scored.index, scored["overall"])
    ax.axhline(75, linestyle="--", color="green", linewidth=1)
    ax.axhline(55, linestyle="--", color="orange", linewidth=1)
    ax.axhline(35, linestyle="--", color="red", linewidth=1)
    ax.set_title("Financial health score (latest fiscal year) - rule-based, not a prediction")
    ax.set_ylabel("score (0-100)"); plt.xticks(rotation=90); plt.tight_layout(); plt.show()

In [ ]:
# 62. Explain ONE company's score indicator by indicator (full transparency)

example = next((t for t in ("AAPL", "MSFT") if t in health_reports), next(iter(health_reports)))
report = health_reports[example]
print(f"{example}: overall {report['overall_score']:.1f} -> {report['grade']}  "
      f"(statements as of {report['as_of_fiscal_year_end'].date()})")
display(report["indicators"].round(3))
print("Indicators with an empty score were unavailable and were ignored (not counted as zero).")

---

# PART 5 — Deployment Readiness

Goal: files that a **Python API** and a **separate frontend** can use later.

```
valuation_pipeline.py      <- data download, cleaning, point-in-time features, preprocessing
company_health.py          <- rule-based health score
valuation_predictor.py     <- load model + analyse a ticker (this cell writes it)
artifacts/
   valuation_model.joblib  <- fitted scikit-learn Pipeline (preprocessing + model)
   model_card.json         <- features, target, training window, metrics, library versions, limits
   supported_companies.csv <- companies whose data actually loaded and were used in training
   training_dataset.csv    <- the exact rows the model was trained on
```

Design choices:

* The saved object is a **single scikit-learn Pipeline**, so imputation, clipping, scaling and encoding are applied automatically at prediction time with the statistics learned in training.
* The final model is **refitted on all rows (train + validation + test)** after evaluation, to use every observation. The metrics in the model card are the *test* metrics from the model before this last refit, so they describe the procedure, not this exact fitted object.
* **Only companies that actually loaded are listed as supported.** Other tickers are analysed on demand if Yahoo has data, but are flagged as *not in the training set*.
* Pickled models are tied to library versions. The versions are stored in the model card; use the same versions in the API environment.

In [ ]:
%%writefile valuation_predictor.py
"""
valuation_predictor.py
======================
Prediction-time code for a future Python API.

    from valuation_predictor import load_bundle, analyze_company
    bundle = load_bundle("artifacts")
    result = analyze_company("AAPL", bundle)      # JSON-friendly dict

It re-uses exactly the same feature code (valuation_pipeline.py) that was used for training,
so training and prediction cannot drift apart.
"""

import json
import math
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

import company_health as ch
import valuation_pipeline as vp

DISCLAIMER = ("The model estimates a typical MARKET CAPITALIZATION for a company with these "
              "fundamentals, based on the companies and years it was trained on. It is not an "
              "intrinsic (DCF) value, not a price target and not investment advice.")


def load_bundle(model_dir="artifacts"):
    """Load the fitted pipeline and its metadata."""
    model_dir = Path(model_dir)
    model_path, card_path = model_dir / "valuation_model.joblib", model_dir / "model_card.json"
    for path in (model_path, card_path):
        if not path.exists():
            raise FileNotFoundError(f"Missing artifact: {path}. Run the export cell of the notebook first.")
    with open(card_path) as handle:
        card = json.load(handle)
    return {"model": joblib.load(model_path), "card": card}


def to_jsonable(obj):
    """Recursively convert numpy/pandas objects to plain JSON types (NaN/inf -> None)."""
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, pd.DataFrame):
        return to_jsonable(obj.to_dict(orient="records"))
    if isinstance(obj, (pd.Timestamp,)):
        return obj.strftime("%Y-%m-%d")
    if isinstance(obj, (np.floating, float)):
        return None if not math.isfinite(float(obj)) else float(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


def snapshot_from_raw(raw, adjust_shares_for_splits=True):
    """Features + current market cap for the LATEST available fiscal year."""
    fund = vp.fundamentals_table(raw)
    cur = fund.iloc[-1]
    if len(fund) < vp.MIN_HISTORY:
        raise vp.DataUnavailableError("need at least two fiscal years of statements")
    if not vp.core_data_ok(cur):
        raise vp.DataUnavailableError("latest statements lack revenue, total assets or share count")
    close = vp.close_series(raw["prices"])
    fye = fund.index[-1]
    factor = vp.split_factor_after(raw.get("splits"), fye) if adjust_shares_for_splits else 1.0
    shares_used = float(cur["shares"]) * factor
    price = float(close.iloc[-1])
    return {
        "fund": fund,
        "features": vp.compute_features(fund),
        "fiscal_year_end": fye,
        "price": price,
        "price_date": close.index[-1],
        "shares_used": shares_used,
        "market_cap": price * shares_used,
        "sector": (raw.get("info") or {}).get("sector") or "Unknown",
    }


def analyze_raw(raw, bundle):
    """Health analysis + valuation estimate from already-downloaded raw data."""
    card, model = bundle["card"], bundle["model"]
    numeric, categorical = card["numeric_features"], card["categorical_features"]
    snap = snapshot_from_raw(raw, card.get("adjust_shares_for_splits", True))

    row = {c: snap["features"].get(c, np.nan) for c in numeric}
    for c in categorical:
        row[c] = snap["sector"]
    X = pd.DataFrame([row], columns=numeric + categorical)
    missing = [c for c in numeric if not np.isfinite(row[c])]

    estimate = float(np.exp(model.predict(X)[0]))
    health = ch.health_report(snap["fund"])

    warnings = []
    if snap["sector"] not in card.get("training_sectors", []):
        warnings.append(f"Sector '{snap['sector']}' was not in the training data.")
    age_days = (pd.Timestamp(snap["price_date"]) - snap["fiscal_year_end"]).days
    if age_days > 540:
        warnings.append(f"Latest annual statements are {age_days} days old.")
    if missing:
        warnings.append("Missing inputs filled with training medians: " + ", ".join(missing))
    if raw["ticker"] not in card.get("training_tickers", []):
        warnings.append("This company was not in the training set; accuracy for unseen companies "
                        "is only indicated by the held-out test results, not guaranteed.")

    return to_jsonable({
        "status": "ok",
        "ticker": raw["ticker"],
        "name": (raw.get("info") or {}).get("longName") or (raw.get("info") or {}).get("shortName"),
        "sector": snap["sector"],
        "in_training_universe": raw["ticker"] in card.get("training_tickers", []),
        "statements_as_of": snap["fiscal_year_end"],
        "price": snap["price"], "price_date": snap["price_date"],
        "current_market_cap_usd": snap["market_cap"],
        "model_estimated_market_cap_usd": estimate,
        "estimate_to_market_ratio": estimate / snap["market_cap"],
        "typical_error": {
            "median_abs_pct_error_on_test": card.get("test_metrics", {}).get("median abs % error"),
            "note": "Median absolute percentage error of the chosen model on a held-out later time period.",
        },
        "health": {
            "overall_score": health["overall_score"], "grade": health["grade"],
            "category_scores": health["category_scores"],
            "indicators": health["indicators"],
        },
        "warnings": warnings,
        "disclaimer": DISCLAIMER,
    })


def analyze_company(ticker, bundle, cache_dir="data_cache", refresh=False):
    """Search-by-ticker entry point. Never raises for data problems: returns a status instead."""
    ticker = (ticker or "").upper().strip()
    if not ticker:
        return {"status": "error", "message": "Please provide a ticker symbol."}
    try:
        raw = vp.fetch_raw(ticker, cache_dir=cache_dir, refresh=refresh)
    except vp.DataUnavailableError as exc:
        return {"status": "unavailable", "ticker": ticker, "message": str(exc)}
    ok, message = vp.check_currency(raw)
    if not ok:
        return {"status": "unsupported", "ticker": ticker,
                "message": f"Not supported: company {message}."}
    try:
        return analyze_raw(raw, bundle)
    except vp.DataUnavailableError as exc:
        return {"status": "unavailable", "ticker": ticker, "message": str(exc)}

## 5.1 Fit the final model and save the artifacts

In [ ]:
# 63. Refit the chosen pipeline on all data and export the artifacts

import platform
import sys
import joblib
import sklearn
import shutil
import zipfile

ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

deploy_model_name, deploy_feature_set = best_key
deploy_numeric = FEATURE_SETS[deploy_feature_set]
deploy_params = candidates[best_key]["best_params"]

deploy_pipeline = clone(make_pipeline(deploy_model_name, deploy_numeric)).set_params(**deploy_params)
deploy_pipeline.fit(dataset[deploy_numeric + CATEGORICAL], dataset[TARGET])        # all rows

in_memory_preds = deploy_pipeline.predict(dataset[deploy_numeric + CATEGORICAL])
joblib.dump(deploy_pipeline, ARTIFACT_DIR / "valuation_model.joblib")

chosen_test = {k: float(v) for k, v in test_rows[best_key].items() if k != "pred"}
model_card = {
    "name": "AI Company Health & Valuation Predictor - valuation model",
    "created_at": pd.Timestamp.now().isoformat(timespec="seconds"),
    "target": "natural log of market capitalization (price x shares) on the prediction date; "
              "exp() of the output gives USD",
    "not_modelled": ["enterprise value", "intrinsic (DCF) value", "future price"],
    "model": deploy_model_name,
    "feature_set": deploy_feature_set,
    "hyperparameters": {k: (None if v is None else v) for k, v in deploy_params.items()},
    "numeric_features": deploy_numeric,
    "categorical_features": CATEGORICAL,
    "report_lag_days": vp.REPORT_LAG_DAYS,
    "adjust_shares_for_splits": True,
    "training_rows": int(len(dataset)),
    "training_date_range": [str(dataset["pred_date"].min().date()), str(dataset["pred_date"].max().date())],
    "training_tickers": sorted(dataset["ticker"].unique().tolist()),
    "training_sectors": sorted(dataset["sector"].unique().tolist()),
    "test_metrics": chosen_test,
    "test_rows": int(len(test)),
    "selection_rule": "lowest validation RMSE among Linear Regression / Ridge / Random Forest using the same 12 core financial features",
    "random_seed": RANDOM_SEED,
    "data_source": "Yahoo Finance via the unofficial yfinance package (restated annual statements)",
    "library_versions": {"python": platform.python_version(), "scikit-learn": sklearn.__version__,
                         "pandas": pd.__version__, "numpy": np.__version__, "joblib": joblib.__version__},
    "limitations": [
        "Small dataset (a few annual rows per company); wide uncertainty.",
        "No market-wide level variable; estimates reflect average multiples of the training period.",
        "Test companies largely overlap with training companies.",
        "Survivorship bias and a single market regime.",
        "Statement availability uses an assumed 90-day reporting lag.",
        "Outputs are statistical market-cap benchmarks, not intrinsic values or investment advice.",
    ],
}
with open(ARTIFACT_DIR / "model_card.json", "w") as handle:
    json.dump(model_card, handle, indent=2)

test_table.reset_index().round(4).to_csv(ARTIFACT_DIR / "model_comparison.csv", index=False)

supported = (dataset.groupby("ticker").agg(sector=("sector", "first"), rows=("ticker", "size"),
                                           first_fiscal_year_end=("fiscal_year_end", "min"),
                                           last_fiscal_year_end=("fiscal_year_end", "max")).reset_index())
supported.to_csv(ARTIFACT_DIR / "supported_companies.csv", index=False)
dataset.to_csv(ARTIFACT_DIR / "training_dataset.csv", index=False)

# Bundle everything (code + artifacts) into one zip
with zipfile.ZipFile("valuation_app_bundle.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for py in ("valuation_pipeline.py", "company_health.py", "valuation_predictor.py"):
        zf.write(py)
    for f in ARTIFACT_DIR.iterdir():
        zf.write(f, f"artifacts/{f.name}")

print("Saved:", sorted(p.name for p in ARTIFACT_DIR.iterdir()))
print("Chosen model:", deploy_model_name, "|", deploy_feature_set, "|", deploy_params)
print(f"Supported companies (data actually loaded): {len(supported)}")
print("Requirements for the API environment:",
      f"scikit-learn=={sklearn.__version__}, pandas=={pd.__version__}, numpy=={np.__version__}, "
      f"joblib=={joblib.__version__}, yfinance")

## 5.2 Prove that the saved files work

A model that cannot be reloaded is useless. We (1) reload from disk and check that predictions are identical, (2) run the prediction code on an already-downloaded company, and (3) check that unsupported or unavailable tickers produce clear messages instead of crashing.

In [ ]:
# 64. Reload test and end-to-end prediction

import importlib
import valuation_predictor as vpred

loaded = joblib.load(ARTIFACT_DIR / "valuation_model.joblib")
reloaded_preds = loaded.predict(dataset[deploy_numeric + CATEGORICAL])
assert np.allclose(reloaded_preds, in_memory_preds), "Reloaded model gives different predictions!"
print("Round trip OK: the reloaded model reproduces the in-memory predictions exactly.")

bundle = vpred.load_bundle(ARTIFACT_DIR)
demo_ticker = "AAPL" if "AAPL" in raw_data else next(iter(raw_data))
result = vpred.analyze_company(demo_ticker, bundle, cache_dir=CACHE_DIR)        # uses the cached download

print(f"\n--- {demo_ticker} ---")
print(json.dumps({k: result[k] for k in ("status", "ticker", "sector", "statements_as_of", "price_date",
                                         "current_market_cap_usd", "model_estimated_market_cap_usd",
                                         "estimate_to_market_ratio")}, indent=2))
print("Health:", result["health"]["overall_score"], result["health"]["grade"])
print("Warnings:", result["warnings"])

# Failure handling: an unknown ticker must return a status, not raise
for bad in ("NOT_A_REAL_TICKER_123", ""):
    outcome = vpred.analyze_company(bad, bundle, cache_dir=CACHE_DIR)
    print(f"\n'{bad}' ->", {k: outcome.get(k) for k in ("status", "message")})

## 5.3 Sketch of the future API (not run here)

> **Not executed or tested in this notebook.** This is a starting point for a separate service. Install `fastapi` and `uvicorn`, put the three `.py` files and the `artifacts/` folder next to it, and test it yourself.

```python
# api.py  (run with: uvicorn api:app --reload)
import pandas as pd
from fastapi import FastAPI, HTTPException
from valuation_predictor import load_bundle, analyze_company

app = FastAPI(title="AI Company Health & Valuation API")
bundle = load_bundle("artifacts")
supported = pd.read_csv("artifacts/supported_companies.csv")

@app.get("/companies")
def companies(q: str = ""):
    """Search the companies the model was trained on."""
    hits = supported[supported["ticker"].str.contains(q.upper(), na=False)]
    return hits.to_dict(orient="records")

@app.get("/analyze/{ticker}")
def analyze(ticker: str):
    result = analyze_company(ticker, bundle)
    if result["status"] != "ok":
        raise HTTPException(status_code=404, detail=result.get("message", "no data"))
    return result
```

**What the frontend can rely on** (fields of the JSON returned by `analyze_company`): `status`, `ticker`, `name`, `sector`, `in_training_universe`, `statements_as_of`, `current_market_cap_usd`, `model_estimated_market_cap_usd`, `estimate_to_market_ratio`, `typical_error`, `health` (`overall_score`, `grade`, `category_scores`, `indicators`), `warnings`, `disclaimer`. Always show `warnings` and `disclaimer` to the user.

A company-search box should list `supported_companies.csv`, and clearly separate "in training set" from "data available but not validated".

---
# PART 6 — Optimization Project: Learning a Company-Valuation Model with Gradient Descent

## 6.1 Problem description

**Real-world problem.** Investors, analysts and lenders often need a quick, objective answer to the question
*"Given this company's financial statements, what market value do companies like it usually have?"*
Doing this by hand means weighing many ratios (size, margins, cash flow, debt, growth) against each other, and
people disagree on how much weight each one deserves.

**Our approach.** We let the data decide the weights. Each company-year in our leak-free dataset (Part 2) has
12 core financial features plus its sector, and a known target:
the natural log of its market capitalization. We look for the weight vector **w** and intercept **b** that make
the predictions $\hat y = b + \mathbf w^\top \mathbf x$ as close as possible to the real values.

"As close as possible" is an **optimization problem**. We solve it with **gradient descent written from scratch**
(no ML library does the optimization), study how the learning rate, momentum and mini-batches change the
convergence, add a **constraint** on the size of the weights, and finally test the learned model on companies and
dates it has never seen.

| Item | In this project |
|---|---|
| Decision variables | weights $w_1,\dots,w_d$ (one per feature) and intercept $b$ |
| Data | company-years from Yahoo Finance statements (Part 2), split by time into train / validation / test |
| Objective | minimise prediction error (mean squared error) + a penalty on large weights |
| Constraint | the weights must stay inside a ball: $\lVert\mathbf w\rVert_2 \le c$ |
| Method | gradient descent (plain, momentum, mini-batch) and projected gradient descent |

## 6.2 Mathematical formulation

**Data.** $n$ training rows. Row $i$ has a standardized feature vector $\mathbf x_i\in\mathbb R^d$ and target
$y_i=\ln(\text{market cap}_i)$. Stacking the rows gives the matrix $X\in\mathbb R^{n\times d}$ and vector $\mathbf y$.

**Model.** $\hat y_i = b + \mathbf w^\top\mathbf x_i$, and the prediction error is $e_i=\hat y_i-y_i$.

**Objective function (ridge regression).**

$$\min_{\mathbf w,\,b}\; J(\mathbf w,b)=\frac{1}{2n}\sum_{i=1}^{n}\big(b+\mathbf w^\top\mathbf x_i-y_i\big)^2+\frac{\lambda}{2}\lVert\mathbf w\rVert_2^2$$

The first term is half the mean squared error; the second term (strength $\lambda\ge 0$) discourages large weights.
The intercept $b$ is not penalized.

**Constraint form.** The same problem can be written with an explicit constraint:

$$\min_{\mathbf w,\,b}\;\frac{1}{2n}\lVert X\mathbf w+b\mathbf 1-\mathbf y\rVert^2 \quad\text{subject to}\quad \lVert\mathbf w\rVert_2\le c .$$

By the Lagrange multiplier / KKT conditions, for every $c$ there is a $\lambda$ that gives the same solution — we
verify this numerically in 6.8.

**Gradient (partial derivatives).** With $\mathbf e = X\mathbf w+b\mathbf 1-\mathbf y$:

$$\frac{\partial J}{\partial \mathbf w}=\frac{1}{n}X^\top\mathbf e+\lambda\mathbf w,\qquad
\frac{\partial J}{\partial b}=\frac{1}{n}\sum_{i=1}^{n}e_i .$$

**Gradient descent update** (learning rate $\eta>0$):

$$\mathbf w_{k+1}=\mathbf w_k-\eta\,\frac{\partial J}{\partial \mathbf w},\qquad b_{k+1}=b_k-\eta\,\frac{\partial J}{\partial b}.$$

**Convexity and step size.** Writing $\boldsymbol\theta=(b,\mathbf w)$ and $\tilde X=[\mathbf 1\;X]$, the Hessian is constant:

$$H=\frac1n\tilde X^\top\tilde X+\lambda\,\mathrm{diag}(0,1,\dots,1).$$

$H$ is positive semidefinite, so $J$ is **convex**: any point where the gradient is zero is the global minimum.
Gradient descent is guaranteed to decrease $J$ when $\eta<2/L$, where $L=\lambda_{\max}(H)$.
The ratio $\kappa=\lambda_{\max}/\lambda_{\min}$ (condition number) controls the speed: large $\kappa$ = slow zig-zag.

**Projected gradient descent** (for the constraint): take a normal gradient step, then pull $\mathbf w$ back onto
the ball: if $\lVert\mathbf w\rVert>c$, set $\mathbf w\leftarrow c\,\mathbf w/\lVert\mathbf w\rVert$.

**Momentum (heavy ball):** $\mathbf v_{k+1}=\beta\mathbf v_k-\eta\nabla J(\boldsymbol\theta_k)$, $\boldsymbol\theta_{k+1}=\boldsymbol\theta_k+\mathbf v_{k+1}$.

**Assumptions.** Relationships are linear in the (standardized, log-transformed) features; rows are treated as
independent; statements were public 90 days after year end (Part 2).

## 6.3 Data matrices

We reuse exactly the same chronological split as Part 3 (train = oldest 60 %, validation = next 20 %,
test = newest 20 %). The preprocessing (median fill, outlier clipping, standardization, one-hot sector) is fitted on
the training rows only. Every figure in Part 6 is also saved in `report_figures/` for the report.

In [ ]:
# 65. Part 6 setup: turn the leak-free split into plain NumPy matrices for our own optimizer

from scipy import sparse

GD_FEATURES = FEATURES_BASE                         # 12 core financial features
FIG_DIR = Path("report_figures")
FIG_DIR.mkdir(exist_ok=True)

prep_opt = vp.make_preprocessor(GD_FEATURES, CATEGORICAL, scale=True)
prep_opt.fit(train[GD_FEATURES + CATEGORICAL])          # fitted on TRAINING rows only

def to_matrix(preprocessor, df):
    X = preprocessor.transform(df[GD_FEATURES + CATEGORICAL])
    return X.toarray() if sparse.issparse(X) else np.asarray(X, dtype=float)

X_tr, X_va, X_te = to_matrix(prep_opt, train), to_matrix(prep_opt, val), to_matrix(prep_opt, test)
y_tr = train[TARGET].to_numpy(float)
y_va = val[TARGET].to_numpy(float)
y_te = test[TARGET].to_numpy(float)
opt_names = [n.replace("num__", "").replace("cat__", "") for n in prep_opt.get_feature_names_out()]

print(f"Training matrix X : {X_tr.shape[0]} company-years x {X_tr.shape[1]} columns "
      f"({len(GD_FEATURES)} numeric features + {X_tr.shape[1] - len(GD_FEATURES)} sector columns)")
print(f"Validation rows   : {len(y_va)}")
print(f"Test rows         : {len(y_te)}")
print(f"Target y = ln(market cap); training mean = {y_tr.mean():.3f} "
      f"(about ${np.exp(y_tr.mean()) / 1e9:,.0f}B)")

## 6.4 Gradient descent implemented from scratch

`ridge_loss` is $J(\mathbf w,b)$, `ridge_gradients` returns the two partial derivatives derived in 6.2, and
`gradient_descent` repeats the update rule. The same function also does momentum, mini-batch SGD and projected
(constrained) gradient descent, controlled by its arguments. `exact_ridge` solves $\nabla J=0$ directly
(a linear system) and is used **only to check** the answer of gradient descent.

In [ ]:
# 66. Gradient descent from scratch (NumPy only - no library does the optimization)

def predict(X, w, b):
    return X @ w + b

def ridge_loss(X, y, w, b, lam):
    """J(w, b) = (1/2n) * sum of squared errors + (lam/2) * ||w||^2"""
    n = len(y)
    error = predict(X, w, b) - y
    return (error @ error) / (2 * n) + (lam / 2) * (w @ w)

def ridge_gradients(X, y, w, b, lam):
    """dJ/dw = (1/n) X^T e + lam * w        dJ/db = (1/n) * sum(e)"""
    n = len(y)
    error = predict(X, w, b) - y
    grad_w = (X.T @ error) / n + lam * w
    grad_b = error.mean()
    return grad_w, grad_b

def gradient_descent(X, y, lam=0.0, learning_rate=0.1, n_iter=5000, tol=1e-10,
                     momentum=0.0, max_norm=None, batch_size=None, seed=0):
    """Minimise the ridge loss J(w, b).

    momentum   : 0 = plain gradient descent; e.g. 0.8 = heavy-ball momentum
    max_norm   : if given, project w back onto the ball ||w|| <= max_norm after every step
                 (projected gradient descent = the CONSTRAINED problem)
    batch_size : None = use all rows (batch GD); a number = mini-batch stochastic GD
    Stops when the parameters move less than `tol` in one step (full-batch only).
    Returns w, b and the full-data loss J after every iteration.
    """
    rng = np.random.default_rng(seed)
    n, d = X.shape
    w, b = np.zeros(d), 0.0                     # start from all-zero weights
    vel_w, vel_b = np.zeros(d), 0.0             # momentum "velocity"
    history = [ridge_loss(X, y, w, b, lam)]

    for _ in range(n_iter):
        if batch_size is None:
            Xb, yb = X, y
        else:
            rows = rng.choice(n, size=min(batch_size, n), replace=False)
            Xb, yb = X[rows], y[rows]

        grad_w, grad_b = ridge_gradients(Xb, yb, w, b, lam)
        vel_w = momentum * vel_w - learning_rate * grad_w
        vel_b = momentum * vel_b - learning_rate * grad_b
        new_w, new_b = w + vel_w, b + vel_b

        if max_norm is not None and np.linalg.norm(new_w) > max_norm:      # projection step
            new_w = new_w * (max_norm / np.linalg.norm(new_w))

        step_size = np.sqrt(np.sum((new_w - w) ** 2) + (new_b - b) ** 2)
        w, b = new_w, new_b
        history.append(ridge_loss(X, y, w, b, lam))

        if not np.isfinite(history[-1]) or history[-1] > 1e12:            # diverged - stop
            break
        if batch_size is None and step_size < tol:
            break
    return w, b, np.array(history)

def hessian_matrix(X, lam):
    """H = (1/n) [1 X]^T [1 X] + lam * diag(0, 1, ..., 1)   (first row/column = intercept b)"""
    n = X.shape[0]
    X1 = np.column_stack([np.ones(n), X])
    penalty = lam * np.eye(X1.shape[1])
    penalty[0, 0] = 0.0
    return X1.T @ X1 / n + penalty

def exact_ridge(X, y, lam):
    """Solve grad J = 0 directly: H theta = (1/n) [1 X]^T y. Used only to CHECK gradient descent."""
    n = X.shape[0]
    X1 = np.column_stack([np.ones(n), X])
    theta = np.linalg.lstsq(hessian_matrix(X, lam), X1.T @ y / n, rcond=None)[0]
    return theta[1:], theta[0]

def save_figure(name):
    plt.savefig(FIG_DIR / f"{name}.png", dpi=200, bbox_inches="tight")

print("Optimizer functions defined.")

## 6.5 Checking the calculus: analytic gradient vs numerical gradient

Before trusting the optimizer we check the partial derivatives from 6.2 against the definition of the derivative,
using the central difference $\dfrac{J(\theta+h)-J(\theta-h)}{2h}$ (Part 1A). If the formulas were wrong,
gradient descent would walk in the wrong direction.

In [ ]:
# 67. Gradient check

rng_check = np.random.default_rng(1)
w_check = rng_check.normal(size=X_tr.shape[1])
b_check, lam_check, h = 0.5, 0.1, 1e-6

grad_w, grad_b = ridge_gradients(X_tr, y_tr, w_check, b_check, lam_check)

numeric_w = np.zeros_like(w_check)
for j in range(len(w_check)):
    step = np.zeros_like(w_check)
    step[j] = h
    numeric_w[j] = (ridge_loss(X_tr, y_tr, w_check + step, b_check, lam_check)
                    - ridge_loss(X_tr, y_tr, w_check - step, b_check, lam_check)) / (2 * h)
numeric_b = (ridge_loss(X_tr, y_tr, w_check, b_check + h, lam_check)
             - ridge_loss(X_tr, y_tr, w_check, b_check - h, lam_check)) / (2 * h)

analytic = np.append(grad_w, grad_b)
numeric = np.append(numeric_w, numeric_b)
relative_error = np.linalg.norm(analytic - numeric) / np.linalg.norm(analytic + numeric)

check_table = pd.DataFrame({"analytic": analytic, "numerical": numeric},
                           index=opt_names + ["intercept b"]).head(6)
display(check_table.round(6))
print(f"Relative difference over all {len(analytic)} partial derivatives: {relative_error:.2e}")
assert relative_error < 1e-6, "Gradient formula does not match the numerical derivative!"
print("Gradient check passed: the derived formulas are correct.")

## 6.6 Convexity and the learning rate

We compute the Hessian eigenvalues of our real problem. Theory (6.2): all eigenvalues $\ge 0$ ⇒ convex, and
gradient descent converges for $\eta<2/L$. We test learning rates on both sides of that limit and plot the
**optimality gap** $J(\theta_k)-J^{\star}$ on a log scale, where $J^{\star}$ is the exact minimum.

In [ ]:
# 68. Convexity check and learning-rate experiment on the real training data

LAM_DEMO = 0.05                                          # a fixed penalty just for this experiment
eigenvalues = np.linalg.eigvalsh(hessian_matrix(X_tr, LAM_DEMO))
L_max, mu_min = eigenvalues.max(), eigenvalues.min()
print(f"Hessian eigenvalues: smallest = {mu_min:.4f}, largest L = {L_max:.4f}")
print(f"All eigenvalues >= 0 -> J is convex: {bool(mu_min >= -1e-10)}")
print(f"Condition number kappa = L / mu = {L_max / mu_min:,.1f}")
print(f"Theory: gradient descent converges if learning rate < 2 / L = {2 / L_max:.4f}\n")

w_star, b_star = exact_ridge(X_tr, y_tr, LAM_DEMO)
J_star = ridge_loss(X_tr, y_tr, w_star, b_star, LAM_DEMO)

lr_settings = {"0.1 / L": 0.1, "0.5 / L": 0.5, "1.0 / L": 1.0, "1.9 / L": 1.9, "2.1 / L (too big)": 2.1}
lr_rows = []
plt.figure(figsize=(9, 4.5))
for label, factor in lr_settings.items():
    iters = 60 if factor > 2 else 3000
    w_lr, b_lr, hist = gradient_descent(X_tr, y_tr, LAM_DEMO, learning_rate=factor / L_max, n_iter=iters)
    gap = np.maximum(hist - J_star, 1e-16)
    plt.semilogy(gap, label=f"η = {label}")
    reached = np.where(gap < 1e-8)[0]
    lr_rows.append({"learning rate": label, "η value": factor / L_max,
                    "iterations to gap < 1e-8": int(reached[0]) if len(reached) else "not reached",
                    "final gap": gap[-1],
                    "result": "DIVERGES" if hist[-1] > hist[0] else "converges"})
plt.axhline(1e-8, color="grey", linestyle=":", linewidth=1)
plt.ylim(1e-14, 1e6)
plt.xlabel("iteration k"); plt.ylabel("J(θ_k) − J*  (log scale)")
plt.title("Effect of the learning rate (real training data)")
plt.legend(); plt.grid(True, which="both", alpha=0.3)
save_figure("fig1_learning_rates"); plt.show()

lr_table = pd.DataFrame(lr_rows).set_index("learning rate")
display(lr_table)

## 6.7 Faster variants: momentum and mini-batch stochastic gradient descent

* **Momentum** remembers the previous step, which damps the zig-zag in narrow valleys (large $\kappa$).
  Theory picks $\eta=\dfrac{4}{(\sqrt L+\sqrt\mu)^2}$ and $\beta=\Big(\dfrac{\sqrt\kappa-1}{\sqrt\kappa+1}\Big)^2$.
* **Mini-batch SGD** estimates the gradient from a random subset of rows. Each step is cheaper, but noisy, so with
  a fixed learning rate it hovers near the minimum instead of reaching it exactly.

In [ ]:
# 69. Plain gradient descent vs momentum vs mini-batch SGD

kappa = L_max / mu_min
eta_momentum = 4 / (np.sqrt(L_max) + np.sqrt(mu_min)) ** 2
beta_momentum = ((np.sqrt(kappa) - 1) / (np.sqrt(kappa) + 1)) ** 2
batch = max(8, len(y_tr) // 8)

variants = {
    "Batch GD (η = 1/L)": dict(learning_rate=1 / L_max),
    f"Momentum (η = {eta_momentum:.3f}, β = {beta_momentum:.2f})": dict(learning_rate=eta_momentum,
                                                                        momentum=beta_momentum),
    f"Mini-batch SGD (batch = {batch}, η = 0.5/L)": dict(learning_rate=0.5 / L_max, batch_size=batch),
}
variant_rows = []
plt.figure(figsize=(9, 4.5))
for label, settings in variants.items():
    _, _, hist = gradient_descent(X_tr, y_tr, LAM_DEMO, n_iter=1500, **settings)
    gap = np.maximum(hist - J_star, 1e-16)
    plt.semilogy(gap, label=label)
    reached = np.where(gap < 1e-6)[0]
    variant_rows.append({"method": label,
                         "iterations to gap < 1e-6": int(reached[0]) if len(reached) else "not reached",
                         "gap after last iteration": gap[-1]})
plt.xlabel("iteration k"); plt.ylabel("J(θ_k) − J*  (log scale)")
plt.title("Convergence of three gradient-descent variants")
plt.legend(); plt.grid(True, which="both", alpha=0.3)
save_figure("fig2_variants"); plt.show()
display(pd.DataFrame(variant_rows).set_index("method"))
print(f"Theory for plain GD: error shrinks by about (1 - 1/kappa) = {1 - 1 / kappa:.4f} per step; "
      f"with momentum by about {(np.sqrt(kappa) - 1) / (np.sqrt(kappa) + 1):.4f} per step.")

## 6.8 Choosing the penalty λ, and the constrained version of the problem

**λ (unconstrained form).** For several values of $\lambda$ we run gradient descent on the training rows and
measure the error on the **validation** rows (never the test rows). Small $\lambda$ can over-fit; large $\lambda$
shrinks all weights towards zero and under-fits.

**Constraint $\lVert\mathbf w\rVert\le c$ (constrained form).** We then solve the constrained problem with projected
gradient descent and $\lambda=0$, choosing $c=\lVert\mathbf w_{\lambda^{\star}}\rVert$. The KKT conditions say both
solutions must be identical, and the Lagrange multiplier recovered from the constrained solution,
$\hat\lambda=-\dfrac{\mathbf w^\top\nabla_{\mathbf w}\text{MSE}}{\lVert\mathbf w\rVert^2}$, must equal $\lambda^{\star}$.

In [ ]:
# 70. Tune lambda on the validation set (each fit is done by our gradient descent)

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def fast_gd(X, y, lam, n_iter=20000):
    """Momentum gradient descent with the theory-based step size for this X and lambda."""
    eig = np.linalg.eigvalsh(hessian_matrix(X, lam))
    L, mu = eig.max(), max(eig.min(), 1e-8)
    eta = 4 / (np.sqrt(L) + np.sqrt(mu)) ** 2
    beta = ((np.sqrt(L / mu) - 1) / (np.sqrt(L / mu) + 1)) ** 2
    return gradient_descent(X, y, lam, learning_rate=eta, momentum=beta, n_iter=n_iter)

lambda_grid = np.logspace(-4, 1, 16)
tuning_rows, weight_paths = [], []
for lam in lambda_grid:
    w_l, b_l, hist = fast_gd(X_tr, y_tr, lam)
    tuning_rows.append({"lambda": lam, "train RMSE": rmse(y_tr, predict(X_tr, w_l, b_l)),
                        "validation RMSE": rmse(y_va, predict(X_va, w_l, b_l)),
                        "||w||": np.linalg.norm(w_l), "GD iterations": len(hist) - 1})
    weight_paths.append(w_l)
tuning = pd.DataFrame(tuning_rows)
weight_paths = np.array(weight_paths)

best_row = tuning.loc[tuning["validation RMSE"].idxmin()]
LAM_BEST = float(best_row["lambda"])
print(f"Best lambda on the validation set: {LAM_BEST:.4g} "
      f"(validation RMSE {best_row['validation RMSE']:.3f}, ||w|| = {best_row['||w||']:.3f})")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].semilogx(tuning["lambda"], tuning["train RMSE"], marker="o", label="training")
axes[0].semilogx(tuning["lambda"], tuning["validation RMSE"], marker="o", label="validation")
axes[0].axvline(LAM_BEST, color="grey", linestyle="--", label=f"chosen λ = {LAM_BEST:.3g}")
axes[0].set_xlabel("λ (log scale)"); axes[0].set_ylabel("RMSE of ln(market cap)")
axes[0].set_title("Choosing λ on the validation set"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

numeric_idx = list(range(len(GD_FEATURES)))
top_idx = sorted(numeric_idx, key=lambda j: -abs(weight_paths[np.argmin(abs(lambda_grid - LAM_BEST)), j]))[:6]
for j in top_idx:
    axes[1].semilogx(lambda_grid, weight_paths[:, j], label=opt_names[j])
axes[1].axvline(LAM_BEST, color="grey", linestyle="--")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_xlabel("λ (log scale)"); axes[1].set_ylabel("weight")
axes[1].set_title("Regularization path: weights shrink as λ grows"); axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)
plt.tight_layout(); save_figure("fig3_lambda_tuning"); plt.show()
display(tuning.round(4))

In [ ]:
# 71. The constrained problem: minimise MSE subject to ||w|| <= c, solved by PROJECTED gradient descent

w_pen, b_pen = exact_ridge(X_tr, y_tr, LAM_BEST)         # penalty solution (for comparison only)
c_limit = np.linalg.norm(w_pen)

L0 = np.linalg.eigvalsh(hessian_matrix(X_tr, 0.0)).max()
w_con, b_con, hist_con = gradient_descent(X_tr, y_tr, lam=0.0, learning_rate=1 / L0, momentum=0.9,
                                          max_norm=c_limit, n_iter=100000, tol=1e-12)

grad_mse_w, _ = ridge_gradients(X_tr, y_tr, w_con, b_con, 0.0)
lambda_recovered = -(w_con @ grad_mse_w) / (w_con @ w_con)

print(f"Constraint limit c = ||w_lambda*|| = {c_limit:.4f}")
print(f"Projected GD finished after {len(hist_con) - 1} iterations; ||w|| = {np.linalg.norm(w_con):.4f} "
      f"(constraint is active)")
print(f"Largest difference between constrained and penalized weights: {np.max(np.abs(w_con - w_pen)):.2e}")
print(f"Lagrange multiplier recovered from KKT: {lambda_recovered:.5f}   vs   lambda* = {LAM_BEST:.5f}")

# Validation error for several constraint sizes c
c_values = c_limit * np.array([0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0])
c_rows = []
for c in c_values:
    w_c, b_c, _ = gradient_descent(X_tr, y_tr, lam=0.0, learning_rate=1 / L0, momentum=0.9,
                                   max_norm=c, n_iter=30000, tol=1e-10)
    c_rows.append({"c": c, "validation RMSE": rmse(y_va, predict(X_va, w_c, b_c)),
                   "||w||": np.linalg.norm(w_c)})
c_table = pd.DataFrame(c_rows)

plt.figure(figsize=(8, 4))
plt.plot(c_table["c"], c_table["validation RMSE"], marker="o")
plt.axvline(c_limit, color="grey", linestyle="--", label=f"c = ||w_λ*|| = {c_limit:.2f}")
plt.xlabel("constraint limit c on ||w||"); plt.ylabel("validation RMSE")
plt.title("Constrained problem: tighter limit = simpler model"); plt.legend(); plt.grid(True, alpha=0.3)
save_figure("fig4_constraint"); plt.show()
display(c_table.round(4))

## 6.9 Final model and test-set results

With $\lambda^{\star}$ fixed, we refit on **train + validation** with gradient descent and evaluate **once** on the test set,
next to the scikit-learn Ridge model (closed-form solver, same objective: sklearn's $\alpha = n\lambda$),
the two baselines, and the Part 3 models.

In [ ]:
# 72. Refit on train + validation with gradient descent and evaluate on the untouched test set

train_val = pd.concat([train, val], ignore_index=True)
prep_final = vp.make_preprocessor(GD_FEATURES, CATEGORICAL, scale=True).fit(train_val[GD_FEATURES + CATEGORICAL])
X_full, X_test_final = to_matrix(prep_final, train_val), to_matrix(prep_final, test)
y_full = train_val[TARGET].to_numpy(float)
final_names = [n.replace("num__", "").replace("cat__", "") for n in prep_final.get_feature_names_out()]

w_final, b_final, hist_final = fast_gd(X_full, y_full, LAM_BEST)
pred_gd = predict(X_test_final, w_final, b_final)

sk_ridge = Ridge(alpha=len(y_full) * LAM_BEST).fit(X_full, y_full)        # same objective, closed form
pred_sk = sk_ridge.predict(X_test_final)
print(f"Gradient descent converged in {len(hist_final) - 1} iterations (final J = {hist_final[-1]:.6f})")
print(f"Largest |weight difference| GD vs scikit-learn Ridge: {np.max(np.abs(w_final - sk_ridge.coef_)):.2e}")
print(f"Largest |prediction difference| on test rows:         {np.max(np.abs(pred_gd - pred_sk)):.2e}")

result_rows = {"Ridge trained by OUR gradient descent": evaluate(y_te, pred_gd),
               "Ridge, scikit-learn closed form (check)": evaluate(y_te, pred_sk)}
for name, fn in BASELINES.items():
    result_rows[name] = evaluate(y_te, fn(train_val, test))
gd_results = pd.DataFrame(result_rows).T

if "test_table" in globals():                       # the Part 3 models, if Part 3 was run
    part3 = test_table.copy()
    part3.index = [f"Part 3: {m} ({f})" for m, f in part3.index]
    part3 = part3[~part3.index.str.contains("Baseline")]
    gd_results = pd.concat([gd_results, part3[gd_results.columns]])

print("\nTEST SET (newest 20 % of dates, never used for training or tuning):")
display(gd_results.sort_values("RMSE (log)").round(3))

## 6.10 Interpreting the learned weights

Because the target is $\ln(\text{market cap})$ and the features are standardized, a weight $w_j$ means:
*a one-standard-deviation increase in feature $j$, with everything else fixed, multiplies the predicted market cap by
$e^{w_j}$.* This turns the optimizer's output into statements an analyst can read.

In [ ]:
# 73. What did gradient descent learn?

weights = pd.Series(w_final, index=final_names)
numeric_weights = weights[GD_FEATURES]
top_weights = numeric_weights.reindex(numeric_weights.abs().sort_values(ascending=False).index).head(10)

interpretation = pd.DataFrame({
    "weight w": top_weights.round(4),
    "effect of +1 std on market cap": [f"{100 * (np.exp(v) - 1):+.1f} %" for v in top_weights],
    })
display(interpretation)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
shown = top_weights.iloc[::-1]
axes[0].barh(shown.index, shown.values, color=["#4c6ef5" if v > 0 else "#e8590c" for v in shown.values])
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set_title("Largest learned weights (standardized features)"); axes[0].set_xlabel("weight")

lims = [min(y_te.min(), pred_gd.min()) - 0.3, max(y_te.max(), pred_gd.max()) + 0.3]
axes[1].scatter(y_te, pred_gd, s=25, alpha=0.8)
axes[1].plot(lims, lims, "r--", label="perfect prediction")
axes[1].set_xlim(lims); axes[1].set_ylim(lims)
axes[1].set_xlabel("actual ln(market cap)"); axes[1].set_ylabel("predicted ln(market cap)")
axes[1].set_title("Test set: gradient-descent model"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); save_figure("fig5_weights_and_predictions"); plt.show()

examples = test[["ticker", "pred_date", "market_cap"]].copy()
examples["predicted market cap"] = np.exp(pred_gd)
examples["error (%)"] = 100 * (examples["predicted market cap"] / examples["market_cap"] - 1)
examples = examples.assign(**{"actual (B USD)": (examples["market_cap"] / 1e9).round(1),
                              "predicted (B USD)": (examples["predicted market cap"] / 1e9).round(1),
                              "error (%)": examples["error (%)"].round(1)})
print("Sample of test predictions:")
display(examples[["ticker", "pred_date", "actual (B USD)", "predicted (B USD)", "error (%)"]]
        .sample(min(8, len(examples)), random_state=RANDOM_SEED))

## 6.11 Seeing gradient descent move: a two-weight version

To draw the loss surface we keep only two strongly related features, **log revenue** and **log total assets**
(intercept fixed at the mean of $y$, which is its optimal value for centered features). Their strong correlation
makes the contours long, narrow ellipses — a high condition number — which is exactly why plain gradient descent
zig-zags and momentum helps.

In [ ]:
# 74. Contour plot of J(w1, w2) with gradient-descent paths

i1, i2 = opt_names.index("log_revenue"), opt_names.index("log_total_assets")
X2 = X_tr[:, [i1, i2]]
X2 = X2 - X2.mean(axis=0)                     # centered, so the best intercept is exactly mean(y)
y2 = y_tr - y_tr.mean()

def loss_2d(w):
    e = X2 @ w - y2
    return (e @ e) / (2 * len(y2))

def grad_2d(w):
    return X2.T @ (X2 @ w - y2) / len(y2)

H2 = X2.T @ X2 / len(y2)
eig2 = np.linalg.eigvalsh(H2)
w2_star = np.linalg.lstsq(H2, X2.T @ y2 / len(y2), rcond=None)[0]
print(f"Correlation between the two features: {np.corrcoef(X2.T)[0, 1]:.3f}")
print(f"2-D Hessian eigenvalues: {eig2.round(4)}  -> condition number {eig2.max() / max(eig2.min(), 1e-12):,.1f}")
print(f"Optimum: w_log_revenue = {w2_star[0]:.3f}, w_log_total_assets = {w2_star[1]:.3f}")

eigvals2, eigvecs2 = np.linalg.eigh(H2)
flat_dir, steep_dir = eigvecs2[:, 0], eigvecs2[:, 1]          # valley direction and steep direction
START_2D = w2_star + 2.5 * flat_dir + 0.6 * steep_dir           # start away from the minimum

def path_2d(eta, beta=0.0, steps=60):
    w, v = START_2D.copy(), np.zeros(2)
    path = [w.copy()]
    for _ in range(steps):
        v = beta * v - eta * grad_2d(w)
        w = w + v
        path.append(w.copy())
    return np.array(path)

L2 = eig2.max()
paths = {"GD, η = 0.5/L (slow along the valley)": path_2d(0.5 / L2),
         "GD, η = 1.9/L (zig-zag across the valley)": path_2d(1.9 / L2),
         "GD with momentum (η = 1/L, β = 0.8)": path_2d(1.0 / L2, beta=0.8)}

all_points = np.vstack(list(paths.values()) + [w2_star[None, :]])
centre = (all_points.max(axis=0) + all_points.min(axis=0)) / 2
half = (all_points.max(axis=0) - all_points.min(axis=0)).max() / 2 + 0.4
g1, g2 = np.meshgrid(np.linspace(centre[0] - half, centre[0] + half, 300),
                     np.linspace(centre[1] - half, centre[1] + half, 300))
Z = np.array([loss_2d(np.array([a, c])) for a, c in zip(g1.ravel(), g2.ravel())]).reshape(g1.shape)
J_min_2d = loss_2d(w2_star)
levels = J_min_2d + (Z.max() - J_min_2d) * np.geomspace(1e-4, 1, 22)

plt.figure(figsize=(8, 7))
plt.contour(g1, g2, Z, levels=levels, cmap="viridis", linewidths=0.8)
for label, p in paths.items():
    end_gap = loss_2d(p[-1]) - J_min_2d
    plt.plot(p[:, 0], p[:, 1], marker=".", markersize=4, linewidth=1.6,
             label=f"{label}: J − J* = {end_gap:.1e} after {len(p) - 1} steps")
plt.plot(*START_2D, "ko", markersize=7, label="start")
plt.plot(*w2_star, "r*", markersize=16, label="minimum")
plt.gca().set_aspect("equal")
plt.xlabel("w (log revenue)"); plt.ylabel("w (log total assets)")
plt.title("Gradient descent on the valuation loss surface")
plt.legend(fontsize=8); save_figure("fig6_contour_paths"); plt.show()

## 6.12 Worked example for the board (viva practice)

Three companies, **one** standardized feature $x$ (think: revenue size), target $y=\ln$(market cap) (illustrative
numbers), $\lambda=0.1$, $\eta=0.5$, start at $w=0,b=0$:

| company | $x_i$ | $y_i$ |
|---|---|---|
| A | −1 | 1 |
| B | 0 | 2 |
| C | 1 | 4 |

Formulas: $e_i=b+wx_i-y_i$, $\;\dfrac{\partial J}{\partial w}=\dfrac13\sum x_ie_i+\lambda w$, $\;\dfrac{\partial J}{\partial b}=\dfrac13\sum e_i$.

**Iteration 1:** $e=(-1,-2,-4)$. $\partial J/\partial w=\tfrac13(1+0-4)+0=-1$, $\partial J/\partial b=\tfrac13(-7)=-2.333$.
$w=0-0.5(-1)=0.5$, $b=0-0.5(-2.333)=1.1667$.

**Iteration 2:** $\hat y=(0.6667,1.1667,1.6667)$, $e=(-0.3333,-0.8333,-2.3333)$.
$\partial J/\partial w=\tfrac13(0.3333-2.3333)+0.1(0.5)=-0.6167$, $\partial J/\partial b=-1.1667$.
$w=0.5+0.3083=0.8083$, $b=1.1667+0.5833=1.75$.

**Exact answer** (set both derivatives to 0; $\sum x_i=0$): $b^{\star}=\bar y=7/3=2.333$,
$w^{\star}=\dfrac{\tfrac13\sum x_iy_i}{\tfrac13\sum x_i^2+\lambda}=\dfrac{1}{0.6667+0.1}=1.3043$.
The code below repeats these numbers and shows the iterations approaching $(w^{\star},b^{\star})$.

In [ ]:
# 75. Board example - every number of the hand calculation

x_small = np.array([-1.0, 0.0, 1.0])
y_small = np.array([1.0, 2.0, 4.0])
lam_small, eta_small = 0.1, 0.5
w_s, b_s = 0.0, 0.0
rows_small = []
for k in range(1, 26):
    e = b_s + w_s * x_small - y_small
    dw = np.mean(x_small * e) + lam_small * w_s
    db = np.mean(e)
    J = np.mean(e ** 2) / 2 + lam_small / 2 * w_s ** 2
    w_s, b_s = w_s - eta_small * dw, b_s - eta_small * db
    rows_small.append({"iteration": k, "errors e": np.round(e, 4), "J before step": round(J, 4),
                       "dJ/dw": round(dw, 4), "dJ/db": round(db, 4), "new w": round(w_s, 4), "new b": round(b_s, 4)})
board = pd.DataFrame(rows_small).set_index("iteration")
display(board.loc[[1, 2, 3, 5, 10, 25]])

w_exact_small = np.mean(x_small * y_small) / (np.mean(x_small ** 2) + lam_small)
print(f"Exact solution: w* = {w_exact_small:.4f}, b* = {y_small.mean():.4f}")

## 6.13 Results summary (generated from the numbers above)

Re-run this cell after any change; copy its output into the *Results and Discussion* section of the report.

In [ ]:
# 76. Automatic results summary for the report

gd_test = gd_results.loc["Ridge trained by OUR gradient descent"]
best_baseline = gd_results.loc[[n for n in BASELINES]].sort_values("RMSE (log)").iloc[0]
best_baseline_name = gd_results.loc[[n for n in BASELINES]].sort_values("RMSE (log)").index[0]
lines = [
    f"1. The loss is convex (smallest Hessian eigenvalue {mu_min:.4f} >= 0), so gradient descent finds the global minimum.",
    f"2. The analytic gradient matched the numerical derivative (relative difference {relative_error:.1e}).",
    f"3. Learning rate: stable below 2/L = {2 / L_max:.4f}; at 2.1/L the loss "
    f"{'diverged' if lr_table.loc['2.1 / L (too big)', 'result'] == 'DIVERGES' else 'did not diverge (check)'}.",
    f"4. Condition number of the problem: {kappa:,.0f}. Momentum reached a gap of 1e-6 in "
    f"{variant_rows[1]['iterations to gap < 1e-6']} iterations vs {variant_rows[0]['iterations to gap < 1e-6']} for plain GD.",
    f"5. Validation chose lambda = {LAM_BEST:.4g}. The constrained problem ||w|| <= {c_limit:.3f} solved by projected GD "
    f"gave the same weights (max difference {np.max(np.abs(w_con - w_pen)):.1e}) and a Lagrange multiplier of {lambda_recovered:.4f}.",
    f"6. Our gradient-descent model matches scikit-learn's closed-form Ridge (max prediction difference "
    f"{np.max(np.abs(pred_gd - pred_sk)):.1e}).",
    f"7. Test set: RMSE {gd_test['RMSE (log)']:.3f}, R2 {gd_test['R2']:.3f}, median absolute error "
    f"{gd_test['median abs % error']:.1f}% of market cap; best baseline ({best_baseline_name}) RMSE "
    f"{best_baseline['RMSE (log)']:.3f} -> the optimized model "
    f"{'BEATS' if gd_test['RMSE (log)'] < best_baseline['RMSE (log)'] else 'does NOT beat'} the baseline.",
    f"8. Strongest learned drivers: " + ", ".join(f"{f} ({100 * (np.exp(v) - 1):+.0f}% per +1 std)"
                                               for f, v in top_weights.head(3).items()) + ".",
]
print("\n".join(lines))
print(f"\nFigures for the report saved in: {FIG_DIR.resolve()}")
for f in sorted(FIG_DIR.glob("*.png")):
    print("  ", f.name)

## 6.14 Discussion: how gradient descent helps with this problem

* **It turns a vague judgement into a solvable problem.** "How much should margins matter compared with size or
  debt?" becomes "minimise $J(\mathbf w,b)$". Gradient descent finds the weights that the historical data support
  best, and the weights are directly readable (6.10).
* **Calculus guarantees the answer.** The gradient (partial derivatives) gives the direction of steepest increase;
  stepping against it lowers the error. Convexity (positive semidefinite Hessian) means there are no bad local minima,
  so the solution is the global optimum — confirmed by the exact solver and by scikit-learn.
* **The learning rate is not a guess.** The largest Hessian eigenvalue gives the limit $\eta<2/L$; the experiment
  shows convergence below it and divergence above it.
* **Conditioning explains speed.** Financial ratios are correlated (revenue vs assets, margins vs ROA), creating
  narrow valleys. Momentum cuts the number of iterations sharply; mini-batch SGD trades exactness for cheaper steps,
  which matters when the dataset grows to thousands of companies.
* **Constraints control complexity.** Limiting $\lVert\mathbf w\rVert$ (or adding $\lambda\lVert\mathbf w\rVert^2$)
  stops the model from relying on noise; projected gradient descent handles the constraint with one extra line.
* **Why use gradient descent if a closed form exists?** For this size the closed form is fine (we use it as a check),
  but gradient descent scales to large data, works with mini-batches and streaming data, and is the same
  method used to train neural networks — where no closed form exists.
* **Limitations.** A linear model on a few hundred company-years; the test period is short and markets move for
  reasons not in the statements. The output is a typical market cap for similar fundamentals, not a fair value or
  investment advice.

## Appendix A — Mathematical details of gradient descent in this project

**A.1 Derivation of the gradient.** $J=\frac{1}{2n}\sum_i e_i^2+\frac\lambda2\sum_j w_j^2$ with
$e_i=b+\sum_j w_jx_{ij}-y_i$. Since $\partial e_i/\partial w_j=x_{ij}$ and $\partial e_i/\partial b=1$, the
chain rule gives
$$\frac{\partial J}{\partial w_j}=\frac1{2n}\sum_i 2e_i\,x_{ij}+\lambda w_j=\frac1n\sum_i x_{ij}e_i+\lambda w_j,\qquad
\frac{\partial J}{\partial b}=\frac1n\sum_i e_i .$$
In matrix form: $\nabla_{\mathbf w}J=\frac1nX^\top\mathbf e+\lambda\mathbf w$.

**A.2 Hessian and convexity.** Differentiating again: $\dfrac{\partial^2 J}{\partial w_j\partial w_k}=\frac1n\sum_i x_{ij}x_{ik}+\lambda\,[j=k]$,
so $H=\frac1n\tilde X^\top\tilde X+\lambda D$. For any vector $\mathbf v$,
$\mathbf v^\top H\mathbf v=\frac1n\lVert\tilde X\mathbf v\rVert^2+\lambda\sum_{j\ge1}v_j^2\ge0$ ⇒ $H$ positive semidefinite ⇒ $J$ convex.

**A.3 Why $\eta<2/L$.** For a quadratic, the error after one step is $\boldsymbol\theta_{k+1}-\boldsymbol\theta^{\star}=(I-\eta H)(\boldsymbol\theta_k-\boldsymbol\theta^{\star})$.
Along an eigenvector with eigenvalue $\lambda_i$ the error is multiplied by $(1-\eta\lambda_i)$. This shrinks only if
$|1-\eta\lambda_i|<1 \iff 0<\eta<2/\lambda_i$. The tightest condition comes from $L=\lambda_{\max}$. With $\eta=1/L$
the slowest direction shrinks by $(1-\mu/L)=(1-1/\kappa)$ per step — hence large $\kappa$ = slow.

**A.4 Closed form (used as a check).** Setting $\nabla J=0$: $\left(\frac1n\tilde X^\top\tilde X+\lambda D\right)\boldsymbol\theta=\frac1n\tilde X^\top\mathbf y$.

**A.5 Constraint and Lagrange multiplier.** For $\min \text{MSE}(\mathbf w,b)$ s.t. $\lVert\mathbf w\rVert^2\le c^2$,
the Lagrangian is $\mathcal L=\text{MSE}+\frac\lambda2(\lVert\mathbf w\rVert^2-c^2)$. KKT: $\nabla_{\mathbf w}\text{MSE}+\lambda\mathbf w=0$,
$\lambda\ge0$, $\lambda(\lVert\mathbf w\rVert^2-c^2)=0$. The first condition is exactly $\nabla_{\mathbf w}J=0$ of the ridge
objective, so both forms share the solution. Taking the dot product with $\mathbf w$ gives
$\lambda=-\mathbf w^\top\nabla_{\mathbf w}\text{MSE}/\lVert\mathbf w\rVert^2$, the value printed in cell 71.

**A.6 Projection onto the ball.** The closest point to $\mathbf u$ inside $\{\lVert\mathbf w\rVert\le c\}$ is
$\mathbf u$ itself if $\lVert\mathbf u\rVert\le c$, otherwise $c\,\mathbf u/\lVert\mathbf u\rVert$.

**A.7 Relation to scikit-learn.** sklearn's Ridge minimises $\lVert\mathbf y-X\mathbf w-b\rVert^2+\alpha\lVert\mathbf w\rVert^2$.
Dividing by $2n$ gives our $J$ with $\lambda=\alpha/n$, so we compare using $\alpha=n\lambda$.

**A.8 Algorithm (pseudocode).**
```
w = 0, b = 0, v = 0
repeat until the step is tiny:
    e      = X w + b - y
    grad_w = (1/n) X^T e + lambda * w
    grad_b = mean(e)
    v      = beta * v - eta * grad          (beta = 0 for plain gradient descent)
    (w, b) = (w, b) + v
    if ||w|| > c: w = c * w / ||w||         (only for the constrained version)
```

---

# Project Status, What Still Needs Testing, and Next Steps

**What this notebook now contains**

* **Part 6 — the gradient-descent optimization project** (problem, formulation, from-scratch GD, learning-rate / momentum / SGD study, constrained version, test results, board example, Appendix A).
* Parts 1–5: complete calculus syllabus demonstrations, a multi-company point-in-time dataset, baselines + Linear/Ridge/Random Forest models with chronological validation, a calculus-feature ablation, explainability, a rule-based health score and exported artifacts.

**What you must run yourself**

* Run all cells **in order** in Google Colab (menu *Runtime → Run all*). The download step needs internet access to Yahoo Finance and takes a few minutes for ~45 tickers the first time.
* The new cells (Parts 2–5) were **not run on real Yahoo Finance data** before this notebook was delivered, because the environment they were prepared in had no access to it. They were checked for logic and runtime errors on *synthetic* stand-in data only, and no number from that check is reported anywhere. Every result you see will come from *your* run: read the automatic summary (cell 59) before writing conclusions, and expect to fix small data issues (renamed Yahoo rows, rate limits) on the first run.
* If Yahoo changes row names or rate-limits you, the download log lists the affected companies; nothing is silently invented.

**Remaining limitations**

* Small sample and annual-only statements; results will be noisy.
* No market-wide variable, so absolute estimates can be systematically high or low in a different market regime.
* Health score thresholds are general rules, not sector-calibrated, and not validated against outcomes.
* The model estimates market capitalization, not intrinsic value.

**Ideas for improvement (only if you have time)**

* Use SEC EDGAR "company facts" data for 10+ years of statements with real filing dates (more rows, true point-in-time).
* Add quarterly data and trailing-twelve-month features.
* Add sector-relative health thresholds and a market-level control variable.
* Validate on companies *never* seen in training (group-wise split).